# Stage 3 — Data Cleaning, Harmonization and Validation

**Project:** The Real Cost of Being an Ojol Driver: What’s Left After a Day on the Road?

Stage 3 converts the validated Stage 2 standardized evidence into a source-preserving processed analytical layer. It profiles integrity, harmonizes source semantics without changing source values, assesses transformation eligibility, records analytical eligibility, and closes the stage with explicit validation and methodological traceability.

The notebook uses the committed Stage 2 outputs as its upstream source of truth and writes the Stage 3 processed-data and metadata artifacts required by later analysis.


## 1. Stage 2 analytical inputs

**Purpose:** prepare direct read-only access to the private GitHub repository from a fresh Google Colab runtime, lock the analysis to the validated Stage 2 commit, and require all committed Stage 2 inputs before Stage 3 processing begins.

The notebook reads `GITHUB_TOKEN` from Colab Secrets only for repository clone/fetch access. The token is never printed, embedded in a URL, written into the repository, or retained after upstream verification.

If the committed Stage 2 state is unavailable, dirty, or does not match the locked commit, Stage 3 stops. No embedded fallback, reconstructed upstream controls, commit, push, or publication logic is used.


In [1]:
from pathlib import Path
from collections import Counter
import csv
import hashlib
import os
import re
import subprocess

from google.colab import userdata

REPO_NAME = "indonesia-ojol-driver-economics-analysis"
REPO_URL = "https://github.com/Ronaldo-spec/indonesia-ojol-driver-economics-analysis.git"
REPO_BRANCH = "main"
EXPECTED_STAGE2_HEAD = "ba7713b27460952d74f3d977d409dcc26dabbe51"
REPO_DIR = Path("/content") / REPO_NAME

REQUIRED_STAGE2_PATHS = [
    "metadata/stage0_measurement_dictionary.csv",
    "metadata/stage1_source_registry.csv",
    "metadata/stage2_extraction_schema.csv",
    "metadata/stage2_standardized_output_manifest.csv",
    "metadata/stage2_comparability_assessment.csv",
    "metadata/stage2_extraction_validation.csv",
    "metadata/stage2_closure_validation.csv",
    "metadata/stage2_closure_summary.csv",
    "notebooks/02_data_extraction_standardization.ipynb",
]

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError(
        "GITHUB_TOKEN is unavailable in Google Colab Secrets. "
        "Grant this notebook access to the secret and run again."
    )

askpass_path = Path("/content/.git_askpass_stage3_readonly.sh")
askpass_path.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) printf '%s\n' 'x-access-token' ;;
  *Password*) printf '%s\n' "$GITHUB_TOKEN" ;;
  *) printf '\n' ;;
esac
""",
    encoding="utf-8",
)
askpass_path.chmod(0o700)

git_env = os.environ.copy()
git_env.update({
    "GITHUB_TOKEN": github_token,
    "GIT_ASKPASS": str(askpass_path),
    "GIT_TERMINAL_PROMPT": "0",
})

def run_git_readonly(args, cwd=None):
    result = subprocess.run(
        ["git", *args],
        cwd=str(cwd) if cwd else None,
        env=git_env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Git read operation failed: git {' '.join(args)}\n{result.stdout}"
        )
    return result.stdout.strip()

try:
    if REPO_DIR.exists():
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(
                f"{REPO_DIR} already exists but is not a Git repository."
            )

        existing_status = run_git_readonly(
            ["status", "--porcelain"],
            cwd=REPO_DIR,
        )
        if existing_status:
            raise RuntimeError(
                "Existing repository checkout contains local changes. "
                "Use a fresh Colab runtime so Stage 3 starts from the committed upstream state."
            )

        branch = run_git_readonly(
            ["branch", "--show-current"],
            cwd=REPO_DIR,
        )
        if branch != REPO_BRANCH:
            raise RuntimeError(
                f"Expected branch {REPO_BRANCH}, found {branch}."
            )

        run_git_readonly(
            ["fetch", "origin", REPO_BRANCH],
            cwd=REPO_DIR,
        )
    else:
        run_git_readonly([
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO_DIR),
        ])

    branch = run_git_readonly(
        ["branch", "--show-current"],
        cwd=REPO_DIR,
    )
    local_head = run_git_readonly(
        ["rev-parse", "HEAD"],
        cwd=REPO_DIR,
    )
    remote_head = run_git_readonly(
        ["rev-parse", "origin/main"],
        cwd=REPO_DIR,
    )

    if branch != REPO_BRANCH:
        raise RuntimeError(
            f"Expected branch {REPO_BRANCH}, found {branch}."
        )

    if local_head != EXPECTED_STAGE2_HEAD:
        raise RuntimeError(
            "Local repository is not at the validated Stage 2 baseline.\n"
            f"Expected: {EXPECTED_STAGE2_HEAD}\n"
            f"Found:    {local_head}"
        )

    if remote_head != EXPECTED_STAGE2_HEAD:
        raise RuntimeError(
            "Remote main no longer matches the validated Stage 2 baseline.\n"
            f"Expected: {EXPECTED_STAGE2_HEAD}\n"
            f"Found:    {remote_head}"
        )

    missing_stage2_paths = [
        relative_path
        for relative_path in REQUIRED_STAGE2_PATHS
        if not (REPO_DIR / relative_path).is_file()
    ]
    if missing_stage2_paths:
        raise FileNotFoundError(
            "Required committed Stage 2 inputs are missing:\n- "
            + "\n- ".join(missing_stage2_paths)
        )

finally:
    askpass_path.unlink(missing_ok=True)
    git_env.pop("GITHUB_TOKEN", None)
    git_env.pop("GIT_ASKPASS", None)
    git_env.pop("GIT_TERMINAL_PROMPT", None)
    del github_token

print("Stage 2 upstream repository verified.")
print(f"Repository: {REPO_DIR}")
print(f"Branch: {branch}")
print(f"Local HEAD: {local_head}")
print(f"Remote HEAD: {remote_head}")
print(f"Required Stage 2 paths: {len(REQUIRED_STAGE2_PATHS)}/{len(REQUIRED_STAGE2_PATHS)}")


Stage 2 upstream repository verified.
Repository: /content/indonesia-ojol-driver-economics-analysis
Branch: main
Local HEAD: ba7713b27460952d74f3d977d409dcc26dabbe51
Remote HEAD: ba7713b27460952d74f3d977d409dcc26dabbe51
Required Stage 2 paths: 9/9


## 2. Standardized evidence controls

Load the project measurement framework and the standardized observation layer used throughout Stage 3.

In [2]:

METADATA_DIR = REPO_DIR / "metadata"

def read_csv(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

def write_csv(path, rows, columns):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=columns,
            extrasaction="raise",
            lineterminator="\n",
        )
        writer.writeheader()
        writer.writerows(rows)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()

def to_float(value):
    value = (value or "").strip()
    return None if value == "" else float(value)

measurement_dictionary = read_csv(
    METADATA_DIR / "stage0_measurement_dictionary.csv"
)
source_registry = read_csv(
    METADATA_DIR / "stage1_source_registry.csv"
)
stage2_schema = read_csv(
    METADATA_DIR / "stage2_extraction_schema.csv"
)
stage2_manifest = read_csv(
    METADATA_DIR / "stage2_standardized_output_manifest.csv"
)
stage2_comparability = read_csv(
    METADATA_DIR / "stage2_comparability_assessment.csv"
)
stage2_validation = read_csv(
    METADATA_DIR / "stage2_extraction_validation.csv"
)
stage2_closure_validation = read_csv(
    METADATA_DIR / "stage2_closure_validation.csv"
)
stage2_closure_rows = read_csv(
    METADATA_DIR / "stage2_closure_summary.csv"
)

if len(stage2_closure_rows) != 1:
    raise RuntimeError(
        f"Expected one Stage 2 closure row, found {len(stage2_closure_rows)}."
    )

stage2_closure = stage2_closure_rows[0]

if stage2_closure["closure_status"] != "PASS_WITH_CAVEAT":
    raise RuntimeError(
        "Unexpected Stage 2 closure status: "
        + stage2_closure["closure_status"]
    )


closure_failures = [
    row["check_id"]
    for row in stage2_closure_validation
    if row["status"] != "PASS"
]
if closure_failures:
    raise RuntimeError(
        "Stage 2 closure validation is not fully PASS: "
        + ", ".join(closure_failures)
    )

source_manifest_rows = [
    row
    for row in stage2_manifest
    if (row.get("output_path") or "").strip()
]

expected_source_file_count = int(
    stage2_closure["standardized_source_file_count"]
)
expected_observation_count = int(
    stage2_closure["standardized_observation_count"]
)
expected_comparison_count = int(
    stage2_closure["comparability_assessment_count"]
)

if len(source_manifest_rows) != expected_source_file_count:
    raise RuntimeError(
        "Stage 2 source-file manifest count does not match closure metadata."
    )

source_paths = {
    row["source_id"]: REPO_DIR / row["output_path"]
    for row in source_manifest_rows
}

missing_source_paths = [
    str(path.relative_to(REPO_DIR))
    for path in source_paths.values()
    if not path.is_file()
]
if missing_source_paths:
    raise FileNotFoundError(
        "Missing standardized Stage 2 source files: "
        + ", ".join(missing_source_paths)
    )

stage3_rows = []
for source_id, path in source_paths.items():
    for row in read_csv(path):
        stage3_rows.append(row)

if len(stage3_rows) != expected_observation_count:
    raise RuntimeError(
        f"Expected {expected_observation_count} standardized observations, "
        f"found {len(stage3_rows)}."
    )

if len(stage2_comparability) != expected_comparison_count:
    raise RuntimeError(
        f"Expected {expected_comparison_count} comparability assessments, "
        f"found {len(stage2_comparability)}."
    )

print("Stage 2 controls loaded.")
print(f"Closure status: {stage2_closure['closure_status']}")
print(f"Standardized source files: {len(source_paths)}")
print(f"Standardized observations: {len(stage3_rows)}")
print(f"Comparability assessments: {len(stage2_comparability)}")


Stage 2 controls loaded.
Closure status: PASS_WITH_CAVEAT
Standardized source files: 5
Standardized observations: 72
Comparability assessments: 17


## 3. Data integrity profile

Profile file integrity, schema consistency, observation counts, denominator availability, and other structural characteristics relevant to later analysis.

In [3]:

schema_columns = [
    row["field_name"]
    for row in stage2_schema
]

input_integrity_rows = []
initial_hashes = {}

for manifest_row in source_manifest_rows:
    source_id = manifest_row["source_id"]
    relative_path = manifest_row["output_path"]
    path = REPO_DIR / relative_path
    rows = read_csv(path)

    with path.open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as f:
        header = next(csv.reader(f))

    file_hash = sha256_file(path)
    initial_hashes[source_id] = file_hash

    input_integrity_rows.append({
        "source_id": source_id,
        "input_path": relative_path,
        "output_role": manifest_row["output_role"],
        "expected_observation_count": int(
            manifest_row["observation_count"]
        ),
        "actual_observation_count": len(rows),
        "schema_match": str(
            header == schema_columns
        ).lower(),
        "sha256": file_hash,
    })

input_integrity_path = (
    METADATA_DIR
    / "stage3_input_integrity_manifest.csv"
)

write_csv(
    input_integrity_path,
    input_integrity_rows,
    [
        "source_id",
        "input_path",
        "output_role",
        "expected_observation_count",
        "actual_observation_count",
        "schema_match",
        "sha256",
    ],
)

profile_rows = []

def add_profile(group, key, count, note):
    profile_rows.append({
        "profile_group": group,
        "profile_key": str(key),
        "count": int(count),
        "note": note,
    })

for key, count in sorted(
    Counter(
        row["source_id"]
        for row in stage3_rows
    ).items()
):
    add_profile(
        "source_id",
        key,
        count,
        "Standardized observation count."
    )

for key, count in sorted(
    Counter(
        row["metric_family"]
        for row in stage3_rows
    ).items()
):
    add_profile(
        "metric_family",
        key,
        count,
        "Source-preserving metric family count."
    )

for key, count in sorted(
    Counter(
        row["mapping_status"]
        for row in stage3_rows
    ).items()
):
    add_profile(
        "mapping_status",
        key,
        count,
        "Stage 2 source-to-project mapping status."
    )

for key, count in sorted(
    Counter(
        row["temporal_evidence_status"]
        for row in stage3_rows
    ).items()
):
    add_profile(
        "temporal_evidence_status",
        key,
        count,
        "Temporal evidence classification."
    )

for key, count in sorted(
    Counter(
        row["value_provenance"]
        for row in stage3_rows
    ).items()
):
    add_profile(
        "value_provenance",
        key,
        count,
        "Evidence provenance classification."
    )

resolved_denominator_count = sum(
    bool(
        (row["metric_denominator_n"] or "").strip()
    )
    for row in stage3_rows
)

unresolved_denominator_count = (
    len(stage3_rows)
    - resolved_denominator_count
)

add_profile(
    "metric_denominator_n",
    "reported_or_resolved",
    resolved_denominator_count,
    "Observation-specific denominator is present."
)

add_profile(
    "metric_denominator_n",
    "unresolved_or_not_reported",
    unresolved_denominator_count,
    "Missing denominator remains missing and is not replaced by sample_size."
)

profile_path = (
    METADATA_DIR
    / "stage3_cleaning_profile.csv"
)

write_csv(
    profile_path,
    profile_rows,
    [
        "profile_group",
        "profile_key",
        "count",
        "note",
    ],
)

print(f"Wrote {input_integrity_path.relative_to(REPO_DIR)}")
print(f"Wrote {profile_path.relative_to(REPO_DIR)}")
print(f"Unresolved metric-specific denominators: {unresolved_denominator_count}")


Wrote metadata/stage3_input_integrity_manifest.csv
Wrote metadata/stage3_cleaning_profile.csv
Unresolved metric-specific denominators: 24


## 4. Data integrity validation

Validate identifiers, required fields, numerical ranges, denominator handling, provenance, comparability constraints, and source-value consistency.

In [4]:

validation_rows = []

def add_check(
    check_id,
    check_name,
    condition,
    severity,
    evidence,
    implication,
    caveat=False,
):
    status = (
        "CAVEAT"
        if condition and caveat
        else "PASS"
        if condition
        else "FAIL"
    )

    validation_rows.append({
        "check_id": check_id,
        "check_name": check_name,
        "status": status,
        "severity": severity,
        "evidence": evidence,
        "analytical_implication": implication,
    })

required_columns = {
    row["field_name"]
    for row in stage2_schema
    if row["required"].strip().lower() == "yes"
}

numeric_columns = {
    row["field_name"]
    for row in stage2_schema
    if row["data_type"].strip().lower() == "number"
}

locked_dictionary_fields = {
    row["field_name"]
    for row in measurement_dictionary
}

registered_source_ids = {
    row["source_id"]
    for row in source_registry
}

# S3B001 — source files.
add_check(
    "S3B001",
    "Expected Stage 2 standardized source files are present",
    len(source_paths) == expected_source_file_count,
    "blocking",
    f"Expected={expected_source_file_count}; found={len(source_paths)}",
    "Stage 3 requires the complete validated Stage 2 source-file set."
)

# S3B002 — manifest row counts.
count_mismatches = [
    row["source_id"]
    for row in input_integrity_rows
    if row["expected_observation_count"]
    != row["actual_observation_count"]
]
add_check(
    "S3B002",
    "Stage 2 manifest row counts are preserved",
    not count_mismatches,
    "blocking",
    f"Mismatches={count_mismatches}",
    "Stage 3 must not silently add or remove standardized observations."
)

# S3B003 — total observations.
add_check(
    "S3B003",
    "Validated Stage 2 observation count is preserved",
    len(stage3_rows) == expected_observation_count,
    "blocking",
    f"Observed={len(stage3_rows)}; expected={expected_observation_count}",
    "The Stage 2 evidence baseline must remain stable."
)

# S3B004 — unique IDs.
duplicate_ids = sorted(
    observation_id
    for observation_id, count
    in Counter(
        row["observation_id"]
        for row in stage3_rows
    ).items()
    if count > 1
)
add_check(
    "S3B004",
    "Observation identifiers remain unique",
    not duplicate_ids,
    "blocking",
    f"Duplicate IDs={duplicate_ids}",
    "Duplicate identifiers would break source-to-analysis lineage."
)

# S3B005 — source registry.
unregistered_sources = sorted({
    row["source_id"]
    for row in stage3_rows
    if row["source_id"]
    not in registered_source_ids
})
add_check(
    "S3B005",
    "All observations resolve to registered sources",
    not unregistered_sources,
    "blocking",
    f"Unregistered source IDs={unregistered_sources}",
    "Every observation must retain canonical source provenance."
)

# S3B006 — schema.
schema_mismatches = [
    row["source_id"]
    for row in input_integrity_rows
    if row["schema_match"] != "true"
]
add_check(
    "S3B006",
    "Standardized files retain the locked Stage 2 schema",
    not schema_mismatches,
    "blocking",
    f"Schema mismatches={schema_mismatches}",
    "Semantic harmonization requires the common source-preserving schema."
)

# S3B007 — required fields.
missing_required = []
for row in stage3_rows:
    for column in required_columns:
        if not (row.get(column) or "").strip():
            missing_required.append(
                f"{row['observation_id']}:{column}"
            )
add_check(
    "S3B007",
    "Required standardized fields remain populated",
    not missing_required,
    "blocking",
    f"Missing required instances={len(missing_required)}",
    "Stage 3 must not infer missing required identity, semantic, or traceability fields."
)

# S3B008 — numeric parseability.
numeric_errors = []
for row in stage3_rows:
    for column in numeric_columns:
        raw = (row.get(column) or "").strip()
        if not raw:
            continue
        try:
            float(raw)
        except ValueError:
            numeric_errors.append(
                f"{row['observation_id']}:{column}"
            )
add_check(
    "S3B008",
    "Populated numeric fields remain parseable",
    not numeric_errors,
    "blocking",
    f"Numeric parse errors={numeric_errors}",
    "Malformed numeric fields would invalidate later analysis."
)

# S3B009 — percent range.
invalid_percentages = []
for row in stage3_rows:
    if row["unit"] == "percent":
        value = to_float(row["value_numeric"])
        if value is None or value < 0 or value > 100:
            invalid_percentages.append(
                row["observation_id"]
            )
add_check(
    "S3B009",
    "Percentage observations remain within 0 to 100",
    not invalid_percentages,
    "blocking",
    f"Invalid percentages={invalid_percentages}",
    "Out-of-range percentages indicate corrupted or incorrectly normalized evidence."
)

# S3B010 — category bounds.
invalid_bounds = []
for row in stage3_rows:
    lower = to_float(row["category_lower_bound"])
    upper = to_float(row["category_upper_bound"])
    if (
        lower is not None
        and upper is not None
        and lower > upper
    ):
        invalid_bounds.append(
            row["observation_id"]
        )
add_check(
    "S3B010",
    "Category bounds remain internally ordered",
    not invalid_bounds,
    "blocking",
    f"Invalid bounds={invalid_bounds}",
    "Distribution brackets must retain valid lower and upper ordering."
)

# S3B011 — denominators.
invalid_denominators = []
for row in stage3_rows:
    denominator = to_float(
        row["metric_denominator_n"]
    )
    sample_size = to_float(
        row["sample_size"]
    )
    if denominator is None:
        continue
    if (
        denominator <= 0
        or (
            sample_size is not None
            and denominator > sample_size
        )
    ):
        invalid_denominators.append(
            row["observation_id"]
        )
add_check(
    "S3B011",
    "Reported metric denominators remain valid",
    not invalid_denominators,
    "blocking",
    f"Invalid denominators={invalid_denominators}",
    "Metric-specific denominators cannot be non-positive or exceed the source sample."
)

# S3B012 — mapped project metrics.
invalid_project_metrics = sorted({
    row["project_metric"]
    for row in stage3_rows
    if row["project_metric"].strip()
    and row["project_metric"]
    not in locked_dictionary_fields
})
add_check(
    "S3B012",
    "Mapped project metrics remain in the locked Stage 0 dictionary",
    not invalid_project_metrics,
    "blocking",
    f"Invalid project metrics={invalid_project_metrics}",
    "Stage 3 must not introduce undeclared project metrics."
)

# S3B013 — inherited semantic safeguards.
required_prior_semantic_checks = {
    "S2V013",
    "S2V014",
    "S2V015",
    "S2V016",
    "S2V017",
    "S2V018",
}
prior_semantic_rows = {
    row["check_id"]: row
    for row in stage2_validation
    if row["check_id"]
    in required_prior_semantic_checks
}
prior_semantic_failures = sorted(
    check_id
    for check_id in required_prior_semantic_checks
    if (
        check_id not in prior_semantic_rows
        or prior_semantic_rows[check_id]["status"]
        != "PASS"
    )
)
add_check(
    "S3B013",
    "Stage 2 semantic safeguards remain binding",
    not prior_semantic_failures,
    "blocking",
    f"Non-PASS inherited safeguards={prior_semantic_failures}",
    "Ambiguous income, mixed costs, deduction evidence layers, time bases, distance bases, and retrospective recall must remain source-defined."
)

# S3B014 — no transformation in standardized layer.
unexpected_transforms = [
    row["observation_id"]
    for row in stage3_rows
    if row["transformation_applied"] != "none"
]
add_check(
    "S3B014",
    "Standardized observations remain untransformed",
    not unexpected_transforms,
    "blocking",
    f"Unexpected transformed rows={unexpected_transforms}",
    "Inflation and other analytical transformations belong to later documented Stage 3 steps."
)

# S3B015 — nominal money.
nominal_violations = []
for row in stage3_rows:
    monetary = (
        row["currency"] == "IDR"
        or row["unit"].startswith("IDR")
        or row["category_bound_unit"].startswith("IDR")
    )
    if (
        monetary
        and row["nominal_real_status"]
        != "nominal"
    ):
        nominal_violations.append(
            row["observation_id"]
        )
add_check(
    "S3B015",
    "Source monetary observations remain nominal",
    not nominal_violations,
    "blocking",
    f"Nominal-status violations={nominal_violations}",
    "Real-value transformations must create documented derived values rather than overwrite source nominal values."
)

# S3B016 — use-specific comparability at observation level.
premature_comparability = [
    row["observation_id"]
    for row in stage3_rows
    if (
        row["comparability_use"].strip()
        or row["comparability_status"].strip()
    )
]
add_check(
    "S3B016",
    "Comparability remains use-specific rather than globally attached to observations",
    not premature_comparability,
    "blocking",
    f"Premature observation-level comparability rows={premature_comparability}",
    "The same observation may have different eligibility across analytical uses."
)

# S3B017 — comparability status distribution.
allowed_statuses = {
    "directly_comparable",
    "comparable_with_transformation",
    "comparable_with_caveat",
    "context_only",
    "not_comparable",
}
invalid_status_rows = [
    row.get("comparison_id", "")
    for row in stage2_comparability
    if row["comparability_status"]
    not in allowed_statuses
]

comparison_counts = Counter(
    row["comparability_status"]
    for row in stage2_comparability
)

expected_comparison_counts = {
    "directly_comparable":
        int(stage2_closure["directly_comparable_count"]),
    "comparable_with_transformation":
        int(stage2_closure["comparable_with_transformation_count"]),
    "comparable_with_caveat":
        int(stage2_closure["comparable_with_caveat_count"]),
    "context_only":
        int(stage2_closure["context_only_count"]),
    "not_comparable":
        int(stage2_closure["not_comparable_count"]),
}

comparison_distribution_match = all(
    comparison_counts.get(status, 0)
    == expected_count
    for status, expected_count
    in expected_comparison_counts.items()
)

add_check(
    "S3B017",
    "Persisted use-specific comparability assessment is internally consistent",
    (
        not invalid_status_rows
        and comparison_distribution_match
        and len(stage2_comparability)
        == expected_comparison_count
    ),
    "blocking",
    (
        f"Observed={dict(comparison_counts)}; "
        f"expected={expected_comparison_counts}; "
        f"invalid={invalid_status_rows}"
    ),
    "Stage 3 must inherit the validated comparison-specific eligibility classes."
)

# S3B018 — Stage 2 validation baseline.
stage2_validation_counts = Counter(
    row["status"]
    for row in stage2_validation
)
stage2_validation_match = (
    len(stage2_validation)
    == int(stage2_closure["validation_check_count"])
    and stage2_validation_counts.get("PASS", 0)
    == int(stage2_closure["validation_pass_count"])
    and stage2_validation_counts.get("CAVEAT", 0)
    == int(stage2_closure["validation_caveat_count"])
    and stage2_validation_counts.get("FAIL", 0)
    == int(stage2_closure["validation_fail_count"])
)
add_check(
    "S3B018",
    "Stage 2 extraction-validation baseline is reproduced",
    stage2_validation_match,
    "blocking",
    (
        f"Checks={len(stage2_validation)}; "
        f"status_counts={dict(stage2_validation_counts)}"
    ),
    "Stage 3 must begin from the validated Stage 2 evidence state."
)

# S3B019 — Stage 2 closure checks.
add_check(
    "S3B019",
    "Stage 2 closure validation remains fully PASS",
    (
        len(stage2_closure_validation) == 6
        and all(
            row["status"] == "PASS"
            for row in stage2_closure_validation
        )
    ),
    "blocking",
    (
        f"Checks={len(stage2_closure_validation)}; "
        f"failures={closure_failures}"
    ),
    "Stage 3 is authorized only from a valid Stage 2 closure."
)

# S3B020 — denominator caveat.
prior_denominator_row = next(
    row
    for row in stage2_validation
    if row["check_id"] == "S2V025"
)
denominator_match = re.search(
    r"n=(\d+)",
    prior_denominator_row["evidence"]
)
prior_unresolved_n = (
    int(denominator_match.group(1))
    if denominator_match
    else None
)

add_check(
    "S3B020",
    "Unresolved metric-specific denominators remain explicit",
    (
        prior_unresolved_n is not None
        and unresolved_denominator_count
        == prior_unresolved_n
    ),
    "non_blocking",
    (
        f"Stage 2 unresolved={prior_unresolved_n}; "
        f"Stage 3B unresolved={unresolved_denominator_count}"
    ),
    "Analyses requiring respondent counts must exclude or caveat these rows rather than assume the full sample denominator.",
    caveat=True,
)

# S3B021 — SRC013 geography caveat.
src013_text = " ".join(
    " ".join([
        row.get("geography", ""),
        row.get("extraction_notes", ""),
    ])
    for row in stage3_rows
    if row["source_id"] == "SRC013"
)
src013_geography_preserved = (
    "62" in src013_text
    and "67" in src013_text
)

add_check(
    "S3B021",
    "SRC013 62-versus-67 locality discrepancy remains explicit",
    src013_geography_preserved,
    "non_blocking",
    "Both locality counts remain present in SRC013 standardized evidence.",
    "No single locality count may be asserted as certain until the source discrepancy is independently resolved.",
    caveat=True,
)

# S3B022 — no direct comparability.
directly_comparable_count = comparison_counts.get(
    "directly_comparable",
    0
)

add_check(
    "S3B022",
    "No assessed cross-source comparison is directly comparable",
    directly_comparable_count == 0,
    "non_blocking",
    f"Directly comparable assessments={directly_comparable_count}",
    "Later analysis must honor transformation, caveat, contextual restriction, or exclusion rather than indiscriminate pooling.",
    caveat=True,
)

# S3B023 — incomplete unit-economics chain.
present_project_metrics = {
    row["project_metric"]
    for row in stage3_rows
    if row["project_metric"].strip()
}

required_chain_metrics = {
    "driver_gross_service_earnings",
    "driver_side_platform_deduction",
    "driver_receipts_before_operating_cost",
    "fuel_cost",
}

missing_chain_metrics = sorted(
    required_chain_metrics
    - present_project_metrics
)

expected_missing_chain = {
    "driver_receipts_before_operating_cost",
    "driver_side_platform_deduction",
    "fuel_cost",
}

add_check(
    "S3B023",
    "Incomplete observed unit-economics chain remains explicit",
    set(missing_chain_metrics)
    == expected_missing_chain,
    "non_blocking",
    f"Missing standardized project-chain metrics={missing_chain_metrics}",
    "The evidence must not be represented as a complete observed gross-to-net chain.",
    caveat=True,
)

# S3B024 — no project net.
project_net_rows = [
    row["observation_id"]
    for row in stage3_rows
    if row["project_metric"]
    in {
        "net_operating_earnings_cash_basis",
        "net_operating_earnings_economic_basis",
    }
]
add_check(
    "S3B024",
    "Stage 3B does not manufacture project net operating earnings",
    not project_net_rows,
    "blocking",
    f"Project net rows={project_net_rows}",
    "Project net operating earnings require a later defensible reconstruction with explicit assumptions and validated inputs."
)

# S3B025 — byte-level immutability.
changed_sources = []
for source_id, path in source_paths.items():
    if sha256_file(path) != initial_hashes[source_id]:
        changed_sources.append(source_id)

add_check(
    "S3B025",
    "Stage 2 standardized source files remain byte-identical during Stage 3B",
    not changed_sources,
    "blocking",
    f"Changed source IDs={changed_sources}",
    "Stage 3B must not mutate the standardized evidence layer."
)

# S3B026 — no processed dataset.
processed_dir = REPO_DIR / "data" / "processed"
add_check(
    "S3B026",
    "Stage 3B does not create a processed analytical dataset",
    not processed_dir.exists(),
    "blocking",
    f"Processed directory exists={processed_dir.exists()}",
    "Processed analytical outputs belong to Stage 3E after semantic and transformation eligibility gates."
)

# S3B027 — no blocking Stage 2 validation failure.
stage2_blocking_failures = [
    row["check_id"]
    for row in stage2_validation
    if (
        row["status"] == "FAIL"
        and row["severity"] == "blocking"
    )
]
add_check(
    "S3B027",
    "No blocking Stage 2 validation failure is carried into Stage 3",
    not stage2_blocking_failures,
    "blocking",
    f"Blocking Stage 2 failures={stage2_blocking_failures}",
    "Stage 3 input integrity requires a Stage 2 baseline with no blocking extraction-validation failure."
)

status_counts = Counter(
    row["status"]
    for row in validation_rows
)

blocking_failures = [
    row["check_id"]
    for row in validation_rows
    if (
        row["status"] == "FAIL"
        and row["severity"] == "blocking"
    )
]

overall_status = (
    "FAIL"
    if blocking_failures
    else "PASS_WITH_CAVEAT"
    if status_counts.get("CAVEAT", 0) > 0
    else "PASS"
)


validation_path = (
    METADATA_DIR
    / "stage3_cleaning_validation.csv"
)

write_csv(
    validation_path,
    validation_rows,
    [
        "check_id",
        "check_name",
        "status",
        "severity",
        "evidence",
        "analytical_implication",
    ],
)

print("Stage 3B cleaning and integrity gate completed.")
print(f"Checks: {len(validation_rows)}")
print(f"PASS: {status_counts.get('PASS', 0)}")
print(f"CAVEAT: {status_counts.get('CAVEAT', 0)}")
print(f"FAIL: {status_counts.get('FAIL', 0)}")
print(f"Overall status: {overall_status}")


Stage 3B cleaning and integrity gate completed.
Checks: 27
PASS: 23
CAVEAT: 4
FAIL: 0
Overall status: PASS_WITH_CAVEAT


## 5. Data integrity summary

Summarize the integrity checks and retain evidence limitations that remain analytically relevant.

In [5]:

summary_rows = [{
    "stage": "Stage 3B",
    "stage_title": "Cleaning and Integrity Gate",
    "status": overall_status,
    "standardized_source_file_count": len(source_paths),
    "standardized_observation_count": len(stage3_rows),
    "validation_check_count": len(validation_rows),
    "validation_pass_count": status_counts.get("PASS", 0),
    "validation_caveat_count": status_counts.get("CAVEAT", 0),
    "validation_fail_count": status_counts.get("FAIL", 0),
    "blocking_failure_count": len(blocking_failures),
    "key_conclusion": (
        "Stage 2 standardized evidence remains structurally intact, with documented denominator, geography, "
        "comparability, and incomplete unit-economics caveats preserved for semantic harmonization."
    ),
}]

summary_path = (
    METADATA_DIR
    / "stage3_cleaning_validation_summary.csv"
)

write_csv(
    summary_path,
    summary_rows,
    [
        "stage",
        "stage_title",
        "status",
        "standardized_source_file_count",
        "standardized_observation_count",
        "validation_check_count",
        "validation_pass_count",
        "validation_caveat_count",
        "validation_fail_count",
        "blocking_failure_count",
        "key_conclusion",
    ],
)

expected_outputs = [
    input_integrity_path,
    profile_path,
    validation_path,
    summary_path,
]

missing_outputs = [
    str(path.relative_to(REPO_DIR))
    for path in expected_outputs
    if not path.is_file()
]

if missing_outputs:
    raise RuntimeError(
        "Missing Stage 3B outputs: "
        + ", ".join(missing_outputs)
    )

caveat_rows = [
    row
    for row in validation_rows
    if row["status"] == "CAVEAT"
]

print("\n========================================")
print("STAGE 3B VALIDATION RESULTS")
print("========================================")
print(f"Status: {overall_status}")
print(f"Checks: {len(validation_rows)}")
print(f"PASS: {status_counts.get('PASS', 0)}")
print(f"CAVEAT: {status_counts.get('CAVEAT', 0)}")
print(f"FAIL: {status_counts.get('FAIL', 0)}")
print(f"Blocking failures: {len(blocking_failures)}")

print("\nOutputs:")
for path in expected_outputs:
    print(f"- {path.relative_to(REPO_DIR)}")

print("\nBinding caveats:")
for row in caveat_rows:
    print(
        f"- {row['check_id']} — "
        f"{row['check_name']}"
    )

if blocking_failures:
    raise RuntimeError(
        "Stage 3B blocking validation failures: "
        + ", ".join(blocking_failures)
    )

print("\nStage 3B validation summary written.")
print("No processed analytical dataset was created.")



STAGE 3B VALIDATION RESULTS
Status: PASS_WITH_CAVEAT
Checks: 27
PASS: 23
CAVEAT: 4
FAIL: 0
Blocking failures: 0

Outputs:
- metadata/stage3_input_integrity_manifest.csv
- metadata/stage3_cleaning_profile.csv
- metadata/stage3_cleaning_validation.csv
- metadata/stage3_cleaning_validation_summary.csv

Binding caveats:
- S3B020 — Unresolved metric-specific denominators remain explicit
- S3B021 — SRC013 62-versus-67 locality discrepancy remains explicit
- S3B022 — No assessed cross-source comparison is directly comparable
- S3B023 — Incomplete observed unit-economics chain remains explicit

Stage 3B validation summary written.
No processed analytical dataset was created.


# Semantic Harmonization

Evaluate whether standardized observations can be represented through the project measurement framework without changing their source-defined meaning.

## 6. Harmonization methodology

Define the semantic rules used to retain defensible project mappings and preserve source-defined or contextual concepts where direct mapping is not justified.

In [6]:

harmonization_rules = [
    {
        "rule_id": "S3C-R01",
        "rule_name": "Retain defensible locked project metrics",
        "decision_rule": (
            "When Stage 2 project_metric is populated and no semantic safeguard is violated, "
            "retain the locked project metric without changing source value, unit, basis, or provenance."
        ),
        "analytical_implication": (
            "A retained mapping supports later project-metric analysis only within the observation's "
            "time, geography, service, vehicle, evidence, and comparability limits."
        ),
        "intended_report_destination": "Methodology — measurement and harmonization",
    },
    {
        "rule_id": "S3C-R02",
        "rule_name": "Preserve source-defined or ambiguous earnings concepts",
        "decision_rule": (
            "Generic income and source-defined net income remain source-defined when they do not match "
            "the project gross, receipts, or net operating earnings definitions."
        ),
        "analytical_implication": (
            "These observations may provide source-specific context but cannot be relabelled as project gross or net."
        ),
        "intended_report_destination": "Methodology — earnings-layer comparability",
    },
    {
        "rule_id": "S3C-R03",
        "rule_name": "Separate driver-reported and realized deduction evidence",
        "decision_rule": (
            "Driver-reported deduction categories or prevalence remain separate from transaction-level "
            "realized driver deductions and from regulatory or platform-stated fee rules."
        ),
        "analytical_implication": (
            "Reported deduction prevalence cannot be used as an observed realized commission rate."
        ),
        "intended_report_destination": "Methodology — deduction evidence layers",
    },
    {
        "rule_id": "S3C-R04",
        "rule_name": "Exclude mixed work-personal bundles from project operating costs",
        "decision_rule": (
            "A cost bundle that combines work costs with personal or household expenditure remains outside "
            "project operating-cost totals unless components are separately reported."
        ),
        "analytical_implication": (
            "Fuel-plus-food/drink bundles are not project fuel cost and are not included in project net operating earnings."
        ),
        "intended_report_destination": "Methodology — operating-cost boundary",
    },
    {
        "rule_id": "S3C-R05",
        "rule_name": "Preserve working-time basis",
        "decision_rule": (
            "Source-reported generic working time remains working_hours_source_reported and is not relabelled "
            "as online_hours or productive_or_engaged_hours."
        ),
        "analytical_implication": (
            "Per-hour analysis must use only denominators compatible with the reported working-time basis."
        ),
        "intended_report_destination": "Methodology — time denominators",
    },
    {
        "rule_id": "S3C-R06",
        "rule_name": "Preserve distance basis",
        "decision_rule": (
            "Source-reported distance with an unresolved basis remains source-defined and is not relabelled "
            "as total_work_related_distance_km."
        ),
        "analytical_implication": (
            "Ambiguous distance cannot directly drive fuel modelling or project per-km earnings."
        ),
        "intended_report_destination": "Methodology — distance and fuel modelling",
    },
    {
        "rule_id": "S3C-R07",
        "rule_name": "Preserve temporal evidence class",
        "decision_rule": (
            "Retrospectively recalled historical values remain retrospective_recall and are not treated as "
            "independent contemporaneous survey waves."
        ),
        "analytical_implication": (
            "Historical trajectory analysis must identify recall evidence and avoid panel-wave interpretation."
        ),
        "intended_report_destination": "Methodology — temporal comparability",
    },
    {
        "rule_id": "S3C-R08",
        "rule_name": "Keep perception and preference evidence contextual",
        "decision_rule": (
            "Survey perception, preference, and promotional-response evidence remains contextual unless the source "
            "directly measures the economic quantity required by the project."
        ),
        "analytical_implication": (
            "Preference or perception values cannot substitute for realized earnings, deductions, or operating costs."
        ),
        "intended_report_destination": "Methodology — evidence roles",
    },
    {
        "rule_id": "S3C-R09",
        "rule_name": "Retain use-specific comparability",
        "decision_rule": (
            "Comparability is assessed for a defined analytical use and is not converted into a global property "
            "of a standardized observation."
        ),
        "analytical_implication": (
            "Later analysis must inherit directly_comparable, comparable_with_transformation, "
            "comparable_with_caveat, context_only, or not_comparable at the comparison-use level."
        ),
        "intended_report_destination": "Methodology — comparability framework",
    },
    {
        "rule_id": "S3C-R10",
        "rule_name": "Do not manufacture a complete unit-economics chain",
        "decision_rule": (
            "Missing project-chain components remain missing; no residual, imputed, or inferred receipts, "
            "deductions, fuel cost, or project net operating earnings are created in Stage 3C."
        ),
        "analytical_implication": (
            "Observed component analysis remains distinct from later modelled or scenario reconstruction."
        ),
        "intended_report_destination": "Methodology — unit economics and reconstruction",
    },
]

harmonization_rules_path = (
    METADATA_DIR / "stage3_harmonization_rules.csv"
)

write_csv(
    harmonization_rules_path,
    harmonization_rules,
    [
        "rule_id",
        "rule_name",
        "decision_rule",
        "analytical_implication",
        "intended_report_destination",
    ],
)

print(f"Wrote {harmonization_rules_path.relative_to(REPO_DIR)}")
print(f"Rules: {len(harmonization_rules)}")


Wrote metadata/stage3_harmonization_rules.csv
Rules: 10


## 7. Observation-level semantic assessment

Assess each standardized observation for project-metric eligibility, source-defined descriptive use, contextual use, and material semantic caveats.

In [7]:

harmonization_assessment = []

for row in stage3_rows:
    project_metric = (row["project_metric"] or "").strip()
    metric_family = (row["metric_family"] or "").strip()
    evidence_type = (row["evidence_type"] or "").strip().lower()
    earnings_layer = (row["earnings_layer"] or "").strip()
    working_time_basis = (row["working_time_basis"] or "").strip()
    distance_basis = (row["distance_basis"] or "").strip()
    cost_boundary = (row["cost_boundary"] or "").strip()
    temporal_status = (row["temporal_evidence_status"] or "").strip()
    source_metric_code = (row["source_metric_code"] or "").strip()

    semantic_flags = []

    if earnings_layer in {"source_defined_net", "unspecified"}:
        semantic_flags.append("earnings_layer_requires_source_definition")

    if cost_boundary == "mixed_work_personal":
        semantic_flags.append("mixed_work_personal_cost_boundary")

    if "driver_reported_deduction" in source_metric_code:
        semantic_flags.append("driver_reported_deduction_not_realized")

    if working_time_basis == "source_reported_unspecified":
        semantic_flags.append("working_time_basis_source_reported")

    if distance_basis == "source_reported_unspecified":
        semantic_flags.append("distance_basis_source_reported")

    if temporal_status == "retrospective_recall":
        semantic_flags.append("retrospective_recall")

    if "perception" in evidence_type or "preference" in evidence_type:
        semantic_flags.append("contextual_perception_or_preference")

    prohibited_mapping = False

    if earnings_layer == "source_defined_net" and project_metric in {
        "net_operating_earnings_cash_basis",
        "net_operating_earnings_economic_basis",
        "driver_gross_service_earnings",
        "driver_receipts_before_operating_cost",
    }:
        prohibited_mapping = True

    if cost_boundary == "mixed_work_personal" and project_metric in {
        "fuel_cost",
        "maintenance_cost",
        "mobile_data_cost",
        "parking_toll_cost",
        "other_operating_cost",
    }:
        prohibited_mapping = True

    if (
        "driver_reported_deduction" in source_metric_code
        and project_metric in {
            "driver_side_platform_deduction",
            "realized_driver_deduction_rate",
        }
    ):
        prohibited_mapping = True

    if (
        working_time_basis == "source_reported_unspecified"
        and project_metric in {
            "online_hours",
            "productive_or_engaged_hours",
        }
    ):
        prohibited_mapping = True

    if (
        distance_basis == "source_reported_unspecified"
        and project_metric == "total_work_related_distance_km"
    ):
        prohibited_mapping = True

    if prohibited_mapping:
        harmonization_action = "reject_project_metric_mapping"
        harmonized_metric = ""
        semantic_eligibility = "not_eligible_for_project_metric"
        rationale = (
            "The Stage 2 mapping would violate a locked semantic boundary and cannot be retained."
        )

    elif project_metric:
        harmonization_action = "retain_locked_project_metric"
        harmonized_metric = project_metric
        semantic_eligibility = (
            "eligible_with_semantic_caveat"
            if semantic_flags
            else "eligible_as_mapped"
        )
        rationale = (
            "The locked project metric is retained without changing source value, unit, basis, provenance, or scope."
        )

    elif metric_family == "context" or (
        "perception" in evidence_type or "preference" in evidence_type
    ):
        harmonization_action = "retain_context_only"
        harmonized_metric = ""
        semantic_eligibility = "context_only"
        rationale = (
            "The observation remains contextual and is not converted into a project unit-economics metric."
        )

    else:
        harmonization_action = "retain_source_defined_metric"
        harmonized_metric = ""
        semantic_eligibility = "source_defined_only"
        rationale = (
            "No defensible locked project-metric mapping is available; the source-defined metric is preserved."
        )

    harmonization_assessment.append({
        "observation_id": row["observation_id"],
        "source_id": row["source_id"],
        "source_metric_code": row["source_metric_code"],
        "source_metric_label": row["source_metric_label"],
        "metric_family": row["metric_family"],
        "stage2_project_metric": row["project_metric"],
        "harmonization_action": harmonization_action,
        "harmonized_metric": harmonized_metric,
        "semantic_eligibility": semantic_eligibility,
        "semantic_flags": " | ".join(semantic_flags),
        "value_provenance": row["value_provenance"],
        "temporal_evidence_status": row["temporal_evidence_status"],
        "earnings_layer": row["earnings_layer"],
        "working_time_basis": row["working_time_basis"],
        "distance_basis": row["distance_basis"],
        "cost_boundary": row["cost_boundary"],
        "rationale": rationale,
    })

harmonization_assessment_path = (
    METADATA_DIR / "stage3_harmonization_assessment.csv"
)

write_csv(
    harmonization_assessment_path,
    harmonization_assessment,
    [
        "observation_id",
        "source_id",
        "source_metric_code",
        "source_metric_label",
        "metric_family",
        "stage2_project_metric",
        "harmonization_action",
        "harmonized_metric",
        "semantic_eligibility",
        "semantic_flags",
        "value_provenance",
        "temporal_evidence_status",
        "earnings_layer",
        "working_time_basis",
        "distance_basis",
        "cost_boundary",
        "rationale",
    ],
)

assessment_counts = Counter(
    row["semantic_eligibility"]
    for row in harmonization_assessment
)

print(f"Wrote {harmonization_assessment_path.relative_to(REPO_DIR)}")
print(f"Assessed observations: {len(harmonization_assessment)}")
print("Semantic eligibility:")
for status, count in sorted(assessment_counts.items()):
    print(f"- {status}: {count}")


Wrote metadata/stage3_harmonization_assessment.csv
Assessed observations: 72
Semantic eligibility:
- context_only: 14
- eligible_as_mapped: 6
- eligible_with_semantic_caveat: 10
- source_defined_only: 42


## 8. Semantic validation

Validate the treatment of source-defined net income, mixed work-personal costs, deduction evidence layers, working-time bases, distance bases, perception evidence, and retrospective recall.

In [8]:

stage3c_validation = []

def add_stage3c_check(
    check_id,
    check_name,
    condition,
    severity,
    evidence,
    implication,
    caveat=False,
):
    status = (
        "CAVEAT"
        if condition and caveat
        else "PASS"
        if condition
        else "FAIL"
    )

    stage3c_validation.append({
        "check_id": check_id,
        "check_name": check_name,
        "status": status,
        "severity": severity,
        "evidence": evidence,
        "analytical_implication": implication,
    })

# S3C001 — complete observation coverage.
assessment_ids = [
    row["observation_id"]
    for row in harmonization_assessment
]
source_ids = [
    row["observation_id"]
    for row in stage3_rows
]

add_stage3c_check(
    "S3C001",
    "Every Stage 2 standardized observation receives one harmonization assessment",
    (
        len(harmonization_assessment) == len(stage3_rows)
        and Counter(assessment_ids) == Counter(source_ids)
    ),
    "blocking",
    (
        f"Standardized observations={len(stage3_rows)}; "
        f"assessments={len(harmonization_assessment)}"
    ),
    "Stage 3C must preserve complete observation-level lineage."
)

# S3C002 — no duplicate assessment IDs.
duplicate_assessment_ids = sorted(
    observation_id
    for observation_id, count
    in Counter(assessment_ids).items()
    if count > 1
)
add_stage3c_check(
    "S3C002",
    "Harmonization assessments remain unique by observation",
    not duplicate_assessment_ids,
    "blocking",
    f"Duplicate assessment IDs={duplicate_assessment_ids}",
    "Each standardized observation may have only one Stage 3C semantic decision."
)

# S3C003 — retained mapping equals Stage 2 mapping.
retained_mapping_violations = [
    row["observation_id"]
    for row in harmonization_assessment
    if (
        row["harmonization_action"] == "retain_locked_project_metric"
        and row["harmonized_metric"] != row["stage2_project_metric"]
    )
]
add_stage3c_check(
    "S3C003",
    "Retained project metrics exactly match the locked Stage 2 mapping",
    not retained_mapping_violations,
    "blocking",
    f"Violations={retained_mapping_violations}",
    "Stage 3C may retain defensible mappings but cannot silently rename project metrics."
)

# S3C004 — source-defined observations stay unmapped.
source_defined_mapping_violations = [
    row["observation_id"]
    for row in harmonization_assessment
    if (
        row["harmonization_action"] in {
            "retain_source_defined_metric",
            "retain_context_only",
        }
        and row["harmonized_metric"]
    )
]
add_stage3c_check(
    "S3C004",
    "Source-defined and contextual observations remain outside locked project metrics",
    not source_defined_mapping_violations,
    "blocking",
    f"Violations={source_defined_mapping_violations}",
    "Ambiguous or contextual evidence cannot be forced into the project metric dictionary."
)

# S3C005 — no prohibited mappings.
rejected_mapping_rows = [
    row["observation_id"]
    for row in harmonization_assessment
    if row["harmonization_action"] == "reject_project_metric_mapping"
]
add_stage3c_check(
    "S3C005",
    "No Stage 2 mapping violates the Stage 3C semantic safeguards",
    not rejected_mapping_rows,
    "blocking",
    f"Rejected Stage 2 mappings={rejected_mapping_rows}",
    "A rejected mapping would require Stage 2 semantic correction before Stage 3D."
)

# S3C006 — driver-reported deduction safeguard.
deduction_semantic_violations = [
    row["observation_id"]
    for row in harmonization_assessment
    if (
        "driver_reported_deduction" in row["source_metric_code"]
        and row["harmonized_metric"] in {
            "driver_side_platform_deduction",
            "realized_driver_deduction_rate",
        }
    )
]
add_stage3c_check(
    "S3C006",
    "Driver-reported deductions remain separate from realized deduction metrics",
    not deduction_semantic_violations,
    "blocking",
    f"Violations={deduction_semantic_violations}",
    "Driver-reported deduction evidence cannot substitute for realized transaction deductions."
)

# S3C007 — mixed cost safeguard.
mixed_cost_semantic_violations = [
    row["observation_id"]
    for row in harmonization_assessment
    if (
        row["cost_boundary"] == "mixed_work_personal"
        and row["harmonized_metric"] in {
            "fuel_cost",
            "maintenance_cost",
            "mobile_data_cost",
            "parking_toll_cost",
            "other_operating_cost",
        }
    )
]
add_stage3c_check(
    "S3C007",
    "Mixed work-personal bundles remain outside project operating costs",
    not mixed_cost_semantic_violations,
    "blocking",
    f"Violations={mixed_cost_semantic_violations}",
    "Personal food/drink expenditure cannot enter project operating-cost totals."
)

# S3C008 — generic working time safeguard.
working_time_semantic_violations = [
    row["observation_id"]
    for row in harmonization_assessment
    if (
        row["working_time_basis"] == "source_reported_unspecified"
        and row["harmonized_metric"] in {
            "online_hours",
            "productive_or_engaged_hours",
        }
    )
]
add_stage3c_check(
    "S3C008",
    "Generic working time remains separate from online and productive time",
    not working_time_semantic_violations,
    "blocking",
    f"Violations={working_time_semantic_violations}",
    "Per-hour economics must preserve the reported time basis."
)

# S3C009 — distance safeguard.
distance_semantic_violations = [
    row["observation_id"]
    for row in harmonization_assessment
    if (
        row["distance_basis"] == "source_reported_unspecified"
        and row["harmonized_metric"] == "total_work_related_distance_km"
    )
]
add_stage3c_check(
    "S3C009",
    "Ambiguous distance remains separate from total work-related distance",
    not distance_semantic_violations,
    "blocking",
    f"Violations={distance_semantic_violations}",
    "Fuel and per-km analysis cannot silently use ambiguous distance as total work distance."
)

# S3C010 — source-defined net safeguard.
source_net_semantic_violations = [
    row["observation_id"]
    for row in harmonization_assessment
    if (
        row["earnings_layer"] == "source_defined_net"
        and row["harmonized_metric"] in {
            "driver_gross_service_earnings",
            "driver_receipts_before_operating_cost",
            "net_operating_earnings_cash_basis",
            "net_operating_earnings_economic_basis",
        }
    )
]
add_stage3c_check(
    "S3C010",
    "Source-defined net income remains separate from project gross, receipts, and net earnings",
    not source_net_semantic_violations,
    "blocking",
    f"Violations={source_net_semantic_violations}",
    "Source-defined net values cannot be relabelled into the project accounting chain."
)

# S3C011 — perception/preference contextual safeguard.
perception_preference_violations = [
    row["observation_id"]
    for row in harmonization_assessment
    if (
        "contextual_perception_or_preference" in row["semantic_flags"]
        and row["semantic_eligibility"] not in {
            "context_only",
            "source_defined_only",
            "eligible_with_semantic_caveat",
        }
    )
]
add_stage3c_check(
    "S3C011",
    "Perception and preference evidence remains contextual or explicitly caveated",
    not perception_preference_violations,
    "blocking",
    f"Violations={perception_preference_violations}",
    "Perception or preference evidence cannot become a realized economic quantity."
)

# S3C012 — retrospective recall preserved.
recall_rows = [
    row
    for row in harmonization_assessment
    if row["temporal_evidence_status"] == "retrospective_recall"
]
recall_flag_violations = [
    row["observation_id"]
    for row in recall_rows
    if "retrospective_recall" not in row["semantic_flags"]
]
add_stage3c_check(
    "S3C012",
    "Retrospective historical observations retain recall classification",
    not recall_flag_violations,
    "blocking",
    f"Recall observations={len(recall_rows)}; violations={recall_flag_violations}",
    "Recalled historical values cannot be interpreted as independent contemporaneous survey waves."
)

# S3C013 — comparability remains use-specific.
observation_level_comparability = [
    row["observation_id"]
    for row in stage3_rows
    if (
        row["comparability_use"].strip()
        or row["comparability_status"].strip()
    )
]
add_stage3c_check(
    "S3C013",
    "Semantic harmonization does not convert use-specific comparability into a global observation property",
    not observation_level_comparability,
    "blocking",
    f"Premature observation-level comparability rows={observation_level_comparability}",
    "Later analytical use must inherit comparison-specific eligibility."
)

# S3C014 — values are untouched.
stage3c_changed_sources = [
    source_id
    for source_id, path in source_paths.items()
    if sha256_file(path) != initial_hashes[source_id]
]
add_stage3c_check(
    "S3C014",
    "Standardized source files remain byte-identical through Stage 3C",
    not stage3c_changed_sources,
    "blocking",
    f"Changed sources={stage3c_changed_sources}",
    "Semantic harmonization must not modify the standardized evidence layer."
)

# S3C015 — no processed dataset.
processed_dir = REPO_DIR / "data" / "processed"
add_stage3c_check(
    "S3C015",
    "Stage 3C does not create a processed analytical dataset",
    not processed_dir.exists(),
    "blocking",
    f"Processed directory exists={processed_dir.exists()}",
    "Processed outputs belong to Stage 3E after transformation eligibility is resolved."
)

# S3C016–S3C019 — inherited binding caveats remain explicit.
inherited_caveat_ids = {"S3B020", "S3B021", "S3B022", "S3B023"}
stage3b_validation_rows = read_csv(
    METADATA_DIR / "stage3_cleaning_validation.csv"
)
stage3b_caveat_map = {
    row["check_id"]: row
    for row in stage3b_validation_rows
    if row["check_id"] in inherited_caveat_ids
}

for idx, check_id in enumerate(sorted(inherited_caveat_ids), start=16):
    present = (
        check_id in stage3b_caveat_map
        and stage3b_caveat_map[check_id]["status"] == "CAVEAT"
    )
    add_stage3c_check(
        f"S3C{idx:03d}",
        f"Inherited Stage 3B caveat {check_id} remains binding",
        present,
        "non_blocking",
        (
            stage3b_caveat_map[check_id]["analytical_implication"]
            if check_id in stage3b_caveat_map
            else "Missing inherited caveat."
        ),
        (
            stage3b_caveat_map[check_id]["analytical_implication"]
            if check_id in stage3b_caveat_map
            else "The Stage 3B evidence limitation must be restored before Stage 3D."
        ),
        caveat=True,
    )

stage3c_status_counts = Counter(
    row["status"]
    for row in stage3c_validation
)

stage3c_blocking_failures = [
    row["check_id"]
    for row in stage3c_validation
    if (
        row["status"] == "FAIL"
        and row["severity"] == "blocking"
    )
]

stage3c_overall_status = (
    "FAIL"
    if stage3c_blocking_failures
    else "PASS_WITH_CAVEAT"
    if stage3c_status_counts.get("CAVEAT", 0) > 0
    else "PASS"
)


stage3c_validation_path = (
    METADATA_DIR / "stage3_harmonization_validation.csv"
)

write_csv(
    stage3c_validation_path,
    stage3c_validation,
    [
        "check_id",
        "check_name",
        "status",
        "severity",
        "evidence",
        "analytical_implication",
    ],
)

print("Stage 3C semantic harmonization gate completed.")
print(f"Checks: {len(stage3c_validation)}")
print(f"PASS: {stage3c_status_counts.get('PASS', 0)}")
print(f"CAVEAT: {stage3c_status_counts.get('CAVEAT', 0)}")
print(f"FAIL: {stage3c_status_counts.get('FAIL', 0)}")
print(f"Overall status: {stage3c_overall_status}")


Stage 3C semantic harmonization gate completed.
Checks: 19
PASS: 15
CAVEAT: 4
FAIL: 0
Overall status: PASS_WITH_CAVEAT


## 9. Semantic assessment summary

Summarize semantic-harmonization results and validation counts.

In [9]:

stage3c_summary_rows = [{
    "stage": "Stage 3C",
    "stage_title": "Semantic Harmonization Gate",
    "status": stage3c_overall_status,
    "standardized_observation_count": len(stage3_rows),
    "harmonization_rule_count": len(harmonization_rules),
    "harmonization_assessment_count": len(harmonization_assessment),
    "validation_check_count": len(stage3c_validation),
    "validation_pass_count": stage3c_status_counts.get("PASS", 0),
    "validation_caveat_count": stage3c_status_counts.get("CAVEAT", 0),
    "validation_fail_count": stage3c_status_counts.get("FAIL", 0),
    "blocking_failure_count": len(stage3c_blocking_failures),
    "key_conclusion": (
        "Stage 3C preserves source-defined concepts when semantic mapping is not defensible, "
        "keeps contextual evidence separate, and retains explicit semantic caveats for mapped observations."
    ),
}]

stage3c_summary_path = (
    METADATA_DIR / "stage3_harmonization_validation_summary.csv"
)

write_csv(
    stage3c_summary_path,
    stage3c_summary_rows,
    [
        "stage",
        "stage_title",
        "status",
        "standardized_observation_count",
        "harmonization_rule_count",
        "harmonization_assessment_count",
        "validation_check_count",
        "validation_pass_count",
        "validation_caveat_count",
        "validation_fail_count",
        "blocking_failure_count",
        "key_conclusion",
    ],
)

expected_stage3c_outputs = [
    harmonization_rules_path,
    harmonization_assessment_path,
    stage3c_validation_path,
    stage3c_summary_path,
]

missing_stage3c_outputs = [
    str(path.relative_to(REPO_DIR))
    for path in expected_stage3c_outputs
    if not path.is_file()
]

if missing_stage3c_outputs:
    raise RuntimeError(
        "Missing Stage 3C outputs: "
        + ", ".join(missing_stage3c_outputs)
    )

print("\n========================================")
print("STAGE 3C VALIDATION RESULTS")
print("========================================")
print(f"Status: {stage3c_overall_status}")
print(f"Checks: {len(stage3c_validation)}")
print(f"PASS: {stage3c_status_counts.get('PASS', 0)}")
print(f"CAVEAT: {stage3c_status_counts.get('CAVEAT', 0)}")
print(f"FAIL: {stage3c_status_counts.get('FAIL', 0)}")
print(f"Blocking failures: {len(stage3c_blocking_failures)}")

print("\nOutputs:")
for path in expected_stage3c_outputs:
    print(f"- {path.relative_to(REPO_DIR)}")

print("\nSemantic eligibility:")
for status, count in sorted(assessment_counts.items()):
    print(f"- {status}: {count}")

if stage3c_blocking_failures:
    raise RuntimeError(
        "Stage 3C blocking validation failures: "
        + ", ".join(stage3c_blocking_failures)
    )

print("\nStage 3C validation summary written.")
print("No standardized source value was changed.")
print("No processed analytical dataset was created.")



STAGE 3C VALIDATION RESULTS
Status: PASS_WITH_CAVEAT
Checks: 19
PASS: 15
CAVEAT: 4
FAIL: 0
Blocking failures: 0

Outputs:
- metadata/stage3_harmonization_rules.csv
- metadata/stage3_harmonization_assessment.csv
- metadata/stage3_harmonization_validation.csv
- metadata/stage3_harmonization_validation_summary.csv

Semantic eligibility:
- context_only: 14
- eligible_as_mapped: 6
- eligible_with_semantic_caveat: 10
- source_defined_only: 42

Stage 3C validation summary written.
No standardized source value was changed.
No processed analytical dataset was created.


# Transformation Eligibility

Evaluate whether transformations required for comparison are supported by compatible reference evidence.

## 10. CPI reference evidence

Register official CPI evidence relevant to monetary-period comparison, including index-base and geographic-coverage changes.

In [10]:

stage3_transformation_references = [
    {
        "reference_id": "S3TREF001",
        "publisher": "BPS-Statistics Indonesia",
        "reference_title": "Consumer Price Indices in 82 Cities in Indonesia (2012=100) 2019",
        "reference_url": "https://www.bps.go.id/en/publication/2020/04/09/91a62fdd238c2b440752e161",
        "release_date": "2020-04-09",
        "coverage_period": "2019",
        "geography": "82 cities; national composite also reported",
        "index_base": "2012=100",
        "evidence_role": "historical_cpi_reference",
        "stage3_use": "Document the pre-2020 CPI base and coverage regime.",
        "use_constraint": "Do not ratio directly to 2018=100 or 2022=100 series without an explicit bridge.",
        "retrieval_date": "2026-08-28",
    },
    {
        "reference_id": "S3TREF002",
        "publisher": "BPS-Statistics Indonesia",
        "reference_title": "Inflation in January 2020; CPI tables using 2018=100",
        "reference_url": "https://www.bps.go.id/en/pressrelease/2020/02/03/1655/inflation-in-january-2020-was-0-39-percent--the-highest-inflation-occured-in-meulaboh-at-1-44-percent.html",
        "release_date": "2020-02-03",
        "coverage_period": "2018-2020 tables",
        "geography": "90 cities; national inflation tables",
        "index_base": "2018=100",
        "evidence_role": "base_transition_and_bridge_reference",
        "stage3_use": "Document the 2012-to-2018 base transition and BPS rebased 2018-2019 CPI table availability.",
        "use_constraint": "Rebased historical tables do not by themselves solve later 2022=100 geography and coverage changes.",
        "retrieval_date": "2026-08-28",
    },
    {
        "reference_id": "S3TREF003",
        "publisher": "BPS-Statistics Indonesia",
        "reference_title": "Consumer Price Index of 90 Cities in Indonesia 2023 (2018=100)",
        "reference_url": "https://www.bps.go.id/en/publication/2024/04/16/6b4474dedd54d8256a4dd746/indeks-harga-konsumen-90-kota-di-indonesia-2023--2018-100-.html",
        "release_date": "2024-04-16",
        "coverage_period": "2023 monthly",
        "geography": "90 regencies/cities; national CPI also reported",
        "index_base": "2018=100",
        "evidence_role": "historical_cpi_reference",
        "stage3_use": "Document the CPI regime covering 2023.",
        "use_constraint": "Do not directly ratio to 2022=100 series; geography must match the driver observation before use.",
        "retrieval_date": "2026-08-28",
    },
    {
        "reference_id": "S3TREF004",
        "publisher": "BPS-Statistics Indonesia",
        "reference_title": "Consumer Price Index of 150 Regencies/Municipalities in Indonesia 2024 (2022=100)",
        "reference_url": "https://www.bps.go.id/en/publication/2025/04/10/760f01390477b1248214ac76/indeks-harga-konsumen-150-kabupaten-kota-di-indonesia-2024-2022-100-.html",
        "release_date": "2025-04-10",
        "coverage_period": "2024 monthly",
        "geography": "150 regencies/municipalities; national CPI also reported",
        "index_base": "2022=100",
        "evidence_role": "current_cpi_regime_reference",
        "stage3_use": "Document the 2024 expansion to 150 areas and adoption of 2022=100.",
        "use_constraint": "Coverage expansion and base-year change must be reconciled before cross-period real-value transformation.",
        "retrieval_date": "2026-08-28",
    },
]

transformation_reference_path = (
    METADATA_DIR / "stage3_transformation_reference_registry.csv"
)

write_csv(
    transformation_reference_path,
    stage3_transformation_references,
    [
        "reference_id",
        "publisher",
        "reference_title",
        "reference_url",
        "release_date",
        "coverage_period",
        "geography",
        "index_base",
        "evidence_role",
        "stage3_use",
        "use_constraint",
        "retrieval_date",
    ],
)

print(f"Wrote {transformation_reference_path.relative_to(REPO_DIR)}")
print(f"Transformation references: {len(stage3_transformation_references)}")


Wrote metadata/stage3_transformation_reference_registry.csv
Transformation references: 4


## 11. Comparison-level transformation assessment

Assess transformation requirements separately for each defined comparison use.

In [11]:

stage2_reference_registry = read_csv(
    METADATA_DIR / "stage2_reference_input_registry.csv"
)

stage3_transformation_eligibility = []

for comparison in stage2_comparability:
    comparison_id = comparison["comparison_id"]
    status = comparison["comparability_status"]
    required_transformation = comparison["required_transformation"]
    comparison_use = comparison["comparison_use"]

    if comparison_id in {"S2C001", "S2C002", "S2C005"}:
        primary_eligibility = "deferred_not_currently_defensible"
        transformation_action = "preserve_nominal_no_primary_real_value"
        transformation_type = "inflation_adjustment"
        evidence_gap = (
            "A defensible observation-matched CPI chain is not yet available across the relevant "
            "period/geography and CPI-regime changes. The project must not direct-ratio 2012=100, "
            "2018=100, and 2022=100 index levels."
        )
        sensitivity_eligibility = "candidate_only_with_explicit_alternative_cpi_assumption"
        rationale = (
            "Inflation adjustment is conceptually required for level comparison, but the current "
            "evidence does not support a single primary transformation that simultaneously resolves "
            "price period, geography, sample scope, and CPI base/coverage transitions."
        )

    elif comparison_id == "S2C017":
        primary_eligibility = "not_a_numeric_transformation"
        transformation_action = "require_scope_and_effective_date_matching_later"
        transformation_type = "regulatory_scope_reconciliation"
        evidence_gap = (
            "Regulatory, platform-stated, and driver-reported deduction layers require service, "
            "effective-date, geography, and evidence-layer reconciliation rather than numerical rescaling."
        )
        sensitivity_eligibility = "not_applicable"
        rationale = (
            "This comparison requires evidence-layer matching, not an arithmetic transformation."
        )

    elif status == "comparable_with_transformation":
        primary_eligibility = "manual_review_required"
        transformation_action = "do_not_transform_until_reviewed"
        transformation_type = "other_required_transformation"
        evidence_gap = (
            "The Stage 2 comparison requires transformation but is not covered by the explicit Stage 3D rules."
        )
        sensitivity_eligibility = "not_assessed"
        rationale = (
            "Unexpected transformed-comparable comparison requires explicit methodological review."
        )

    else:
        primary_eligibility = "no_transformation_required"
        transformation_action = "retain_source_value_and_comparability_status"
        transformation_type = "none"
        evidence_gap = ""
        sensitivity_eligibility = "not_applicable"
        rationale = (
            "The Stage 2 comparability assessment does not require a numerical transformation for this use."
        )

    stage3_transformation_eligibility.append({
        "comparison_id": comparison_id,
        "comparison_use": comparison_use,
        "stage2_comparability_status": status,
        "stage2_required_transformation": required_transformation,
        "transformation_type": transformation_type,
        "primary_transformation_eligibility": primary_eligibility,
        "transformation_action": transformation_action,
        "sensitivity_eligibility": sensitivity_eligibility,
        "evidence_gap": evidence_gap,
        "critical_caveats": comparison["critical_caveats"],
        "allowed_use_after_stage3d": comparison["allowed_use"],
        "prohibited_use_after_stage3d": comparison["prohibited_use"],
        "rationale": rationale,
    })

transformation_eligibility_path = (
    METADATA_DIR / "stage3_transformation_eligibility.csv"
)

write_csv(
    transformation_eligibility_path,
    stage3_transformation_eligibility,
    [
        "comparison_id",
        "comparison_use",
        "stage2_comparability_status",
        "stage2_required_transformation",
        "transformation_type",
        "primary_transformation_eligibility",
        "transformation_action",
        "sensitivity_eligibility",
        "evidence_gap",
        "critical_caveats",
        "allowed_use_after_stage3d",
        "prohibited_use_after_stage3d",
        "rationale",
    ],
)

eligibility_counts = Counter(
    row["primary_transformation_eligibility"]
    for row in stage3_transformation_eligibility
)

print(f"Wrote {transformation_eligibility_path.relative_to(REPO_DIR)}")
print("Primary transformation eligibility:")
for status, count in sorted(eligibility_counts.items()):
    print(f"- {status}: {count}")


Wrote metadata/stage3_transformation_eligibility.csv
Primary transformation eligibility:
- deferred_not_currently_defensible: 3
- no_transformation_required: 13
- not_a_numeric_transformation: 1


## 12. Monetary transformation methodology

Document the treatment of nominal values, CPI base changes, geographic compatibility, and deferred transformations.

In [12]:

stage3d_decisions = [
    {
        "decision_id": "S3D-D01",
        "decision": "Preserve all source monetary observations in nominal terms.",
        "rationale": (
            "Stage 2 source values are nominal evidence and must remain auditable. "
            "Real-value transformations, when eligible, must be separate derived observations."
        ),
        "evidence_basis": (
            "Stage 0 measurement rules; Stage 2 schema nominal_real_status; Stage 3B/3C validation."
        ),
        "analytical_implication": (
            "No source value is overwritten by an inflation-adjusted value."
        ),
        "intended_report_destination": "Methodology — monetary transformations",
    },
    {
        "decision_id": "S3D-D02",
        "decision": "Do not direct-ratio CPI index levels across different BPS base-year regimes.",
        "rationale": (
            "BPS CPI coverage/base regimes changed from 82 cities using 2012=100, "
            "to 90 cities using 2018=100, and later to 150 regencies/municipalities using 2022=100."
        ),
        "evidence_basis": (
            "BPS CPI 2019 publication; January 2020 BPS release; BPS CPI 2023 publication; "
            "BPS CPI 2024 publication."
        ),
        "analytical_implication": (
            "A documented bridge/rebased series and compatible geography are required before real-value level comparison."
        ),
        "intended_report_destination": "Methodology — CPI compatibility",
    },
    {
        "decision_id": "S3D-D03",
        "decision": (
            "Defer primary inflation adjustment for S2C001, S2C002, and S2C005."
        ),
        "rationale": (
            "Although these comparisons require inflation treatment for nominal level comparison, "
            "the currently registered evidence does not provide a single observation-matched CPI chain "
            "that resolves price period, geography, sample coverage, and CPI regime changes."
        ),
        "evidence_basis": (
            "Stage 2 comparability assessment plus Stage 2 and Stage 3 transformation reference registries."
        ),
        "analytical_implication": (
            "Stage 3E may retain nominal/source-defined analytical rows and eligibility flags, "
            "but must not create primary inflation-adjusted values for these comparisons."
        ),
        "intended_report_destination": "Methodology — transformation eligibility",
    },
    {
        "decision_id": "S3D-D04",
        "decision": (
            "Permit only explicitly labelled sensitivity transformations if a defensible alternative CPI assumption is later specified."
        ),
        "rationale": (
            "A sensitivity transformation may be useful for robustness, but it cannot be presented as the primary observed value "
            "when geography or period matching remains imperfect."
        ),
        "evidence_basis": (
            "Project scenario/sensitivity rules and Stage 2 comparison caveats."
        ),
        "analytical_implication": (
            "Any later sensitivity real-value series must retain CPI source, geography, period, formula, and caveat metadata."
        ),
        "intended_report_destination": "Methodology — sensitivity analysis",
    },
    {
        "decision_id": "S3D-D05",
        "decision": (
            "Treat S2C017 as scope/effective-date reconciliation rather than numerical transformation."
        ),
        "rationale": (
            "Regulatory ceilings, platform-stated policies, and driver-reported deductions are separate evidence layers."
        ),
        "evidence_basis": (
            "Stage 0 deduction-layer rules; Stage 2 comparability assessment."
        ),
        "analytical_implication": (
            "No fee percentage is rescaled or substituted across evidence layers."
        ),
        "intended_report_destination": "Methodology — regulatory and deduction reconciliation",
    },
]

stage3d_decision_path = (
    METADATA_DIR / "stage3_transformation_decision_log.csv"
)

write_csv(
    stage3d_decision_path,
    stage3d_decisions,
    [
        "decision_id",
        "decision",
        "rationale",
        "evidence_basis",
        "analytical_implication",
        "intended_report_destination",
    ],
)

print(f"Wrote {stage3d_decision_path.relative_to(REPO_DIR)}")
print(f"Decisions: {len(stage3d_decisions)}")


Wrote metadata/stage3_transformation_decision_log.csv
Decisions: 5


## 13. Transformation validation

Validate comparison coverage, CPI compatibility, source-value preservation, and transformation status.

In [13]:

stage3d_validation = []

def add_stage3d_check(
    check_id,
    check_name,
    condition,
    severity,
    evidence,
    implication,
    caveat=False,
):
    status = (
        "CAVEAT"
        if condition and caveat
        else "PASS"
        if condition
        else "FAIL"
    )

    stage3d_validation.append({
        "check_id": check_id,
        "check_name": check_name,
        "status": status,
        "severity": severity,
        "evidence": evidence,
        "analytical_implication": implication,
    })

# S3D001 — full comparison coverage.
stage2_comparison_ids = [
    row["comparison_id"]
    for row in stage2_comparability
]
stage3d_comparison_ids = [
    row["comparison_id"]
    for row in stage3_transformation_eligibility
]

add_stage3d_check(
    "S3D001",
    "Every Stage 2 comparison receives one transformation-eligibility assessment",
    Counter(stage2_comparison_ids) == Counter(stage3d_comparison_ids),
    "blocking",
    (
        f"Stage 2 comparisons={len(stage2_comparison_ids)}; "
        f"Stage 3D assessments={len(stage3d_comparison_ids)}"
    ),
    "Transformation eligibility must preserve complete comparison-level lineage."
)

# S3D002 — exactly three inflation-required comparisons.
inflation_rows = [
    row
    for row in stage3_transformation_eligibility
    if row["transformation_type"] == "inflation_adjustment"
]

add_stage3d_check(
    "S3D002",
    "The three Stage 2 inflation-required comparisons are explicitly identified",
    {
        row["comparison_id"]
        for row in inflation_rows
    } == {"S2C001", "S2C002", "S2C005"},
    "blocking",
    f"Inflation-required comparison IDs={[row['comparison_id'] for row in inflation_rows]}",
    "Stage 3D must not omit or invent inflation-required comparisons."
)

# S3D003 — all three are deferred for primary use.
non_deferred_inflation = [
    row["comparison_id"]
    for row in inflation_rows
    if row["primary_transformation_eligibility"]
    != "deferred_not_currently_defensible"
]

add_stage3d_check(
    "S3D003",
    "Primary inflation adjustment remains deferred where CPI compatibility is unresolved",
    not non_deferred_inflation,
    "non_blocking",
    f"Non-deferred inflation comparisons={non_deferred_inflation}",
    (
        "Primary real-value comparison is withheld rather than created from an incompatible or insufficiently matched CPI chain."
    ),
    caveat=True,
)

# S3D004 — no direct CPI-ratio authorization.
direct_ratio_authorizations = [
    row["comparison_id"]
    for row in stage3_transformation_eligibility
    if "direct_ratio" in row["transformation_action"]
]

add_stage3d_check(
    "S3D004",
    "No direct CPI ratio across base-year regimes is authorized",
    not direct_ratio_authorizations,
    "blocking",
    f"Direct-ratio authorizations={direct_ratio_authorizations}",
    "2012=100, 2018=100, and 2022=100 index levels must not be ratioed without a documented bridge."
)

# S3D005 — nominal standardized values remain untouched.
stage3d_changed_sources = [
    source_id
    for source_id, path in source_paths.items()
    if sha256_file(path) != initial_hashes[source_id]
]

add_stage3d_check(
    "S3D005",
    "Standardized source files remain byte-identical through Stage 3D",
    not stage3d_changed_sources,
    "blocking",
    f"Changed source IDs={stage3d_changed_sources}",
    "Transformation eligibility assessment must not alter source evidence."
)

# S3D006 — no processed data yet.
processed_dir = REPO_DIR / "data" / "processed"

add_stage3d_check(
    "S3D006",
    "Stage 3D does not create a processed analytical dataset",
    not processed_dir.exists(),
    "blocking",
    f"Processed directory exists={processed_dir.exists()}",
    "Processed analytical outputs belong to Stage 3E."
)

# S3D007 — four official CPI regime references.
reference_bases = {
    row["index_base"]
    for row in stage3_transformation_references
}

add_stage3d_check(
    "S3D007",
    "Transformation-reference registry documents all relevant CPI base regimes",
    {"2012=100", "2018=100", "2022=100"}.issubset(reference_bases),
    "blocking",
    f"Registered CPI bases={sorted(reference_bases)}",
    "Inflation eligibility decisions must document the CPI base transitions they rely on."
)

# S3D008 — Stage 2 CPI references retained.
stage2_cpi_refs = [
    row
    for row in stage2_reference_registry
    if row["reference_type"] == "cpi"
]

add_stage3d_check(
    "S3D008",
    "Stage 2 CPI references remain available as current-period reference evidence",
    len(stage2_cpi_refs) == 2,
    "blocking",
    f"Stage 2 CPI reference count={len(stage2_cpi_refs)}",
    "Stage 3D supplements rather than overwrites the Stage 2 reference registry."
)

# S3D009 — no transformation formula/value created.
transformed_source_rows = [
    row["observation_id"]
    for row in stage3_rows
    if (
        row["transformation_applied"] != "none"
        or row["transformation_formula"].strip()
    )
]

add_stage3d_check(
    "S3D009",
    "No transformed source observation or formula is created in Stage 3D",
    not transformed_source_rows,
    "blocking",
    f"Transformed source rows={transformed_source_rows}",
    "Eligibility decisions must remain separate from derived-value construction."
)

# S3D010 — S2C017 remains non-numeric.
s2c017 = next(
    row
    for row in stage3_transformation_eligibility
    if row["comparison_id"] == "S2C017"
)

add_stage3d_check(
    "S3D010",
    "Regulatory-policy reconciliation remains non-numeric",
    s2c017["primary_transformation_eligibility"]
    == "not_a_numeric_transformation",
    "blocking",
    f"S2C017 eligibility={s2c017['primary_transformation_eligibility']}",
    "Scope and evidence-layer matching must not be replaced by arithmetic fee transformation."
)

# S3D011 — no unexpected manual-review transformed comparison.
manual_review_rows = [
    row["comparison_id"]
    for row in stage3_transformation_eligibility
    if row["primary_transformation_eligibility"]
    == "manual_review_required"
]

add_stage3d_check(
    "S3D011",
    "No transformed-comparable comparison remains outside the explicit Stage 3D rules",
    not manual_review_rows,
    "blocking",
    f"Manual-review comparison IDs={manual_review_rows}",
    "All transformed-comparable uses must have an explicit eligibility decision."
)

# S3D012 — inherited Stage 3C closure.
stage3c_summary_rows = read_csv(
    METADATA_DIR / "stage3_harmonization_validation_summary.csv"
)

stage3c_ok = (
    len(stage3c_summary_rows) == 1
    and stage3c_summary_rows[0]["status"] == "PASS_WITH_CAVEAT"
    and int(stage3c_summary_rows[0]["blocking_failure_count"]) == 0
)

add_stage3d_check(
    "S3D012",
    "Stage 3C semantic gate remains eligible",
    stage3c_ok,
    "blocking",
    (
        stage3c_summary_rows[0]["status"]
        if stage3c_summary_rows
        else "missing"
    ),
    "Transformation eligibility may only proceed from a valid semantic-harmonization gate."
)

# S3D013 — transformation deferral caveat.
add_stage3d_check(
    "S3D013",
    "Primary real-value comparisons remain unavailable for the three inflation-required uses",
    len(inflation_rows) == 3,
    "non_blocking",
    "S2C001, S2C002, and S2C005 remain nominal for primary analysis.",
    (
        "Primary analytical use must not present inflation-adjusted values for these comparisons "
        "unless a later defensible transformation is separately introduced and validated."
    ),
    caveat=True,
)

stage3d_status_counts = Counter(
    row["status"]
    for row in stage3d_validation
)

stage3d_blocking_failures = [
    row["check_id"]
    for row in stage3d_validation
    if (
        row["status"] == "FAIL"
        and row["severity"] == "blocking"
    )
]

stage3d_overall_status = (
    "FAIL"
    if stage3d_blocking_failures
    else "PASS_WITH_CAVEAT"
    if stage3d_status_counts.get("CAVEAT", 0) > 0
    else "PASS"
)


stage3d_validation_path = (
    METADATA_DIR / "stage3_transformation_validation.csv"
)

write_csv(
    stage3d_validation_path,
    stage3d_validation,
    [
        "check_id",
        "check_name",
        "status",
        "severity",
        "evidence",
        "analytical_implication",
    ],
)

print("Stage 3D transformation eligibility gate completed.")
print(f"Checks: {len(stage3d_validation)}")
print(f"PASS: {stage3d_status_counts.get('PASS', 0)}")
print(f"CAVEAT: {stage3d_status_counts.get('CAVEAT', 0)}")
print(f"FAIL: {stage3d_status_counts.get('FAIL', 0)}")
print(f"Overall status: {stage3d_overall_status}")


Stage 3D transformation eligibility gate completed.
Checks: 13
PASS: 11
CAVEAT: 2
FAIL: 0
Overall status: PASS_WITH_CAVEAT


## 14. Transformation assessment summary

Summarize transformation-eligibility results and validation counts.

In [14]:

stage3d_summary_rows = [{
    "stage": "Stage 3D",
    "stage_title": "Transformation Eligibility Gate",
    "status": stage3d_overall_status,
    "comparison_assessment_count": len(stage3_transformation_eligibility),
    "inflation_required_comparison_count": len(inflation_rows),
    "primary_inflation_transformation_count": 0,
    "deferred_inflation_transformation_count": len(inflation_rows),
    "transformation_reference_count": len(stage3_transformation_references),
    "methodological_decision_count": len(stage3d_decisions),
    "validation_check_count": len(stage3d_validation),
    "validation_pass_count": stage3d_status_counts.get("PASS", 0),
    "validation_caveat_count": stage3d_status_counts.get("CAVEAT", 0),
    "validation_fail_count": stage3d_status_counts.get("FAIL", 0),
    "blocking_failure_count": len(stage3d_blocking_failures),
    "key_conclusion": (
        "Transformation eligibility is explicitly assessed for all comparison uses; "
        "no primary inflation-adjusted value is created where the CPI evidence is insufficiently matched."
    ),
}]

stage3d_summary_path = (
    METADATA_DIR / "stage3_transformation_validation_summary.csv"
)

write_csv(
    stage3d_summary_path,
    stage3d_summary_rows,
    [
        "stage",
        "stage_title",
        "status",
        "comparison_assessment_count",
        "inflation_required_comparison_count",
        "primary_inflation_transformation_count",
        "deferred_inflation_transformation_count",
        "transformation_reference_count",
        "methodological_decision_count",
        "validation_check_count",
        "validation_pass_count",
        "validation_caveat_count",
        "validation_fail_count",
        "blocking_failure_count",
        "key_conclusion",
    ],
)

expected_stage3d_outputs = [
    transformation_reference_path,
    transformation_eligibility_path,
    stage3d_decision_path,
    stage3d_validation_path,
    stage3d_summary_path,
]

missing_stage3d_outputs = [
    str(path.relative_to(REPO_DIR))
    for path in expected_stage3d_outputs
    if not path.is_file()
]

if missing_stage3d_outputs:
    raise RuntimeError(
        "Missing Stage 3D outputs: "
        + ", ".join(missing_stage3d_outputs)
    )

print("\n========================================")
print("STAGE 3D VALIDATION RESULTS")
print("========================================")
print(f"Status: {stage3d_overall_status}")
print(f"Checks: {len(stage3d_validation)}")
print(f"PASS: {stage3d_status_counts.get('PASS', 0)}")
print(f"CAVEAT: {stage3d_status_counts.get('CAVEAT', 0)}")
print(f"FAIL: {stage3d_status_counts.get('FAIL', 0)}")
print(f"Blocking failures: {len(stage3d_blocking_failures)}")

print("\nOutputs:")
for path in expected_stage3d_outputs:
    print(f"- {path.relative_to(REPO_DIR)}")

print("\nPrimary transformation eligibility:")
for status, count in sorted(eligibility_counts.items()):
    print(f"- {status}: {count}")

if stage3d_blocking_failures:
    raise RuntimeError(
        "Stage 3D blocking validation failures: "
        + ", ".join(stage3d_blocking_failures)
    )

print("\nStage 3D validation summary written.")
print("No inflation-adjusted primary value was created.")
print("No standardized source value was changed.")
print("No processed analytical dataset was created.")



STAGE 3D VALIDATION RESULTS
Status: PASS_WITH_CAVEAT
Checks: 13
PASS: 11
CAVEAT: 2
FAIL: 0
Blocking failures: 0

Outputs:
- metadata/stage3_transformation_reference_registry.csv
- metadata/stage3_transformation_eligibility.csv
- metadata/stage3_transformation_decision_log.csv
- metadata/stage3_transformation_validation.csv
- metadata/stage3_transformation_validation_summary.csv

Primary transformation eligibility:
- deferred_not_currently_defensible: 3
- no_transformation_required: 13
- not_a_numeric_transformation: 1

Stage 3D validation summary written.
No inflation-adjusted primary value was created.
No standardized source value was changed.
No processed analytical dataset was created.


# Processed Analytical Evidence

Construct a source-preserving analytical evidence register while keeping contextual and non-comparable observations explicitly distinguishable.

## 15. Processed observation register

Combine standardized observations with semantic eligibility while preserving original values, units, provenance, and nominal status.

In [15]:

PROCESSED_DIR = REPO_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

harmonization_by_id = {
    row["observation_id"]: row
    for row in harmonization_assessment
}

processed_rows = []

for row in stage3_rows:
    observation_id = row["observation_id"]

    if observation_id not in harmonization_by_id:
        raise RuntimeError(
            f"Missing Stage 3C harmonization assessment for {observation_id}."
        )

    h = harmonization_by_id[observation_id]

    if h["semantic_eligibility"] in {
        "eligible_as_mapped",
        "eligible_with_semantic_caveat",
    }:
        observation_analysis_role = "project_metric_evidence"

    elif h["semantic_eligibility"] == "context_only":
        observation_analysis_role = "context_evidence"

    elif h["semantic_eligibility"] == "source_defined_only":
        observation_analysis_role = "source_defined_descriptive_evidence"

    else:
        observation_analysis_role = "excluded_pending_semantic_resolution"

    if not row["metric_denominator_n"].strip():
        denominator_status = "metric_specific_denominator_unresolved"
    else:
        denominator_status = "metric_specific_denominator_available"

    processed_rows.append({
        **row,
        "harmonized_metric": h["harmonized_metric"],
        "semantic_eligibility": h["semantic_eligibility"],
        "semantic_flags": h["semantic_flags"],
        "observation_analysis_role": observation_analysis_role,
        "denominator_status": denominator_status,
        "processed_value_numeric": row["value_numeric"],
        "processed_unit": row["unit"],
        "processed_value_provenance": row["value_provenance"],
        "processed_nominal_real_status": row["nominal_real_status"],
        "processed_transformation_status": "none",
        "processed_transformation_formula": "",
        "processed_stage": "Stage 3E",
    })

processed_path = (
    PROCESSED_DIR / "driver_evidence_processed.csv"
)

processed_columns = (
    schema_columns
    + [
        "harmonized_metric",
        "semantic_eligibility",
        "semantic_flags",
        "observation_analysis_role",
        "denominator_status",
        "processed_value_numeric",
        "processed_unit",
        "processed_value_provenance",
        "processed_nominal_real_status",
        "processed_transformation_status",
        "processed_transformation_formula",
        "processed_stage",
    ]
)

write_csv(
    processed_path,
    processed_rows,
    processed_columns,
)

print(f"Wrote {processed_path.relative_to(REPO_DIR)}")
print(f"Processed observations: {len(processed_rows)}")
print("Observation analysis roles:")
for role, count in sorted(
    Counter(
        row["observation_analysis_role"]
        for row in processed_rows
    ).items()
):
    print(f"- {role}: {count}")


Wrote data/processed/driver_evidence_processed.csv
Processed observations: 72
Observation analysis roles:
- context_evidence: 14
- project_metric_evidence: 16
- source_defined_descriptive_evidence: 42


## 16. Comparison-level analytical eligibility

Translate semantic, comparability, and transformation assessments into explicit analytical-use restrictions.

In [16]:

transformation_by_comparison = {
    row["comparison_id"]: row
    for row in stage3_transformation_eligibility
}

analysis_eligibility_rows = []

for comparison in stage2_comparability:
    comparison_id = comparison["comparison_id"]

    if comparison_id not in transformation_by_comparison:
        raise RuntimeError(
            f"Missing Stage 3D transformation assessment for {comparison_id}."
        )

    t = transformation_by_comparison[comparison_id]
    status = comparison["comparability_status"]
    transformation_eligibility = t["primary_transformation_eligibility"]

    if status == "directly_comparable":
        primary_analysis_eligibility = "eligible_direct_comparison"

    elif status == "comparable_with_caveat":
        primary_analysis_eligibility = "eligible_with_caveat"

    elif status == "comparable_with_transformation":
        if transformation_eligibility == "deferred_not_currently_defensible":
            primary_analysis_eligibility = "not_eligible_for_primary_transformed_comparison"
        elif transformation_eligibility == "no_transformation_required":
            primary_analysis_eligibility = "eligible_after_documented_transformation"
        else:
            primary_analysis_eligibility = "conditional_transformation_review"

    elif status == "context_only":
        primary_analysis_eligibility = "context_only"

    elif status == "not_comparable":
        primary_analysis_eligibility = "not_comparable"

    else:
        raise RuntimeError(
            f"Unexpected comparability status for {comparison_id}: {status}"
        )

    analysis_eligibility_rows.append({
        "comparison_id": comparison_id,
        "comparison_use": comparison["comparison_use"],
        "metric_concept": comparison["metric_concept"],
        "source_ids": comparison["source_ids"],
        "observation_ids": comparison["observation_ids"],
        "stage2_comparability_status": status,
        "stage2_required_transformation": comparison["required_transformation"],
        "stage3d_transformation_eligibility": transformation_eligibility,
        "stage3d_transformation_action": t["transformation_action"],
        "primary_analysis_eligibility": primary_analysis_eligibility,
        "allowed_use": comparison["allowed_use"],
        "prohibited_use": comparison["prohibited_use"],
        "critical_caveats": comparison["critical_caveats"],
        "analytical_note": (
            "Eligibility remains specific to this comparison use and must not be generalized "
            "to all observations from the same source or metric family."
        ),
    })

analysis_eligibility_path = (
    METADATA_DIR / "stage3_analysis_eligibility.csv"
)

write_csv(
    analysis_eligibility_path,
    analysis_eligibility_rows,
    [
        "comparison_id",
        "comparison_use",
        "metric_concept",
        "source_ids",
        "observation_ids",
        "stage2_comparability_status",
        "stage2_required_transformation",
        "stage3d_transformation_eligibility",
        "stage3d_transformation_action",
        "primary_analysis_eligibility",
        "allowed_use",
        "prohibited_use",
        "critical_caveats",
        "analytical_note",
    ],
)

analysis_eligibility_counts = Counter(
    row["primary_analysis_eligibility"]
    for row in analysis_eligibility_rows
)

print(f"Wrote {analysis_eligibility_path.relative_to(REPO_DIR)}")
print("Primary analysis eligibility:")
for status, count in sorted(analysis_eligibility_counts.items()):
    print(f"- {status}: {count}")


Wrote metadata/stage3_analysis_eligibility.csv
Primary analysis eligibility:
- context_only: 3
- eligible_with_caveat: 4
- not_comparable: 7
- not_eligible_for_primary_transformed_comparison: 3


## 17. Analytical output manifest

Record the processed evidence register and comparison-level analytical eligibility outputs.

In [17]:

stage3e_output_manifest = [
    {
        "output_id": "S3E-OUT001",
        "output_path": str(processed_path.relative_to(REPO_DIR)),
        "output_type": "processed_observation_register",
        "row_count": len(processed_rows),
        "source_stage": "Stage 3E",
        "role": (
            "Source-preserving processed evidence register with semantic eligibility "
            "and observation-level analysis role."
        ),
    },
    {
        "output_id": "S3E-OUT002",
        "output_path": str(analysis_eligibility_path.relative_to(REPO_DIR)),
        "output_type": "comparison_level_analysis_eligibility",
        "row_count": len(analysis_eligibility_rows),
        "source_stage": "Stage 3E",
        "role": (
            "Use-specific analytical eligibility retaining Stage 2 comparability and "
            "Stage 3D transformation decisions."
        ),
    },
]

stage3e_manifest_path = (
    METADATA_DIR / "stage3_output_manifest.csv"
)

write_csv(
    stage3e_manifest_path,
    stage3e_output_manifest,
    [
        "output_id",
        "output_path",
        "output_type",
        "row_count",
        "source_stage",
        "role",
    ],
)

print(f"Wrote {stage3e_manifest_path.relative_to(REPO_DIR)}")


Wrote metadata/stage3_output_manifest.csv


## 18. Processed-data validation

Validate observation lineage, source-value consistency, semantic roles, comparison restrictions, missingness, and transformation status.

In [18]:

stage3e_validation = []

def add_stage3e_check(
    check_id,
    check_name,
    condition,
    severity,
    evidence,
    implication,
    caveat=False,
):
    status = (
        "CAVEAT"
        if condition and caveat
        else "PASS"
        if condition
        else "FAIL"
    )

    stage3e_validation.append({
        "check_id": check_id,
        "check_name": check_name,
        "status": status,
        "severity": severity,
        "evidence": evidence,
        "analytical_implication": implication,
    })

# S3E001 — processed row preservation.
add_stage3e_check(
    "S3E001",
    "Processed evidence register preserves all standardized observations",
    len(processed_rows) == len(stage3_rows) == expected_observation_count,
    "blocking",
    (
        f"Standardized={len(stage3_rows)}; "
        f"processed={len(processed_rows)}; "
        f"expected={expected_observation_count}"
    ),
    "Stage 3E must retain the complete standardized evidence layer."
)

# S3E002 — unique IDs.
processed_ids = [
    row["observation_id"]
    for row in processed_rows
]
duplicate_processed_ids = sorted(
    observation_id
    for observation_id, count
    in Counter(processed_ids).items()
    if count > 1
)

add_stage3e_check(
    "S3E002",
    "Processed observation identifiers remain unique",
    not duplicate_processed_ids,
    "blocking",
    f"Duplicate processed IDs={duplicate_processed_ids}",
    "Processed lineage requires a one-to-one observation identity."
)

# S3E003 — same observation set.
standardized_ids = {
    row["observation_id"]
    for row in stage3_rows
}
processed_id_set = set(processed_ids)

add_stage3e_check(
    "S3E003",
    "Processed and standardized observation sets are identical",
    standardized_ids == processed_id_set,
    "blocking",
    (
        f"Missing in processed={sorted(standardized_ids - processed_id_set)}; "
        f"unexpected in processed={sorted(processed_id_set - standardized_ids)}"
    ),
    "Stage 3E may not add synthetic observations or drop evidence."
)

# S3E004 — source values preserved.
standardized_by_id = {
    row["observation_id"]: row
    for row in stage3_rows
}

value_preservation_violations = []
for row in processed_rows:
    source = standardized_by_id[row["observation_id"]]

    checks = [
        row["value_numeric"] == source["value_numeric"],
        row["unit"] == source["unit"],
        row["processed_value_numeric"] == source["value_numeric"],
        row["processed_unit"] == source["unit"],
        row["source_value_text"] == source["source_value_text"],
    ]

    if not all(checks):
        value_preservation_violations.append(
            row["observation_id"]
        )

add_stage3e_check(
    "S3E004",
    "Processed rows preserve standardized source values and units",
    not value_preservation_violations,
    "blocking",
    f"Violations={value_preservation_violations}",
    "Processed analytical preparation must not alter the underlying source-reported values."
)

# S3E005 — no primary real transformation.
real_or_transformed_rows = [
    row["observation_id"]
    for row in processed_rows
    if (
        row["processed_transformation_status"] != "none"
        or row["processed_transformation_formula"].strip()
        or row["processed_nominal_real_status"] != row["nominal_real_status"]
    )
]

add_stage3e_check(
    "S3E005",
    "Stage 3E creates no unsupported transformed primary values",
    not real_or_transformed_rows,
    "blocking",
    f"Transformed rows={real_or_transformed_rows}",
    "Deferred Stage 3D inflation transformations must remain deferred."
)

# S3E006 — semantic counts reproduce Stage 3C.
processed_semantic_counts = Counter(
    row["semantic_eligibility"]
    for row in processed_rows
)

add_stage3e_check(
    "S3E006",
    "Processed semantic eligibility reproduces the Stage 3C assessment",
    processed_semantic_counts == assessment_counts,
    "blocking",
    (
        f"Processed={dict(processed_semantic_counts)}; "
        f"Stage 3C={dict(assessment_counts)}"
    ),
    "Stage 3E may not silently strengthen or weaken semantic eligibility."
)

# S3E007 — project metric evidence only where mapped.
project_metric_role_violations = [
    row["observation_id"]
    for row in processed_rows
    if (
        row["observation_analysis_role"] == "project_metric_evidence"
        and not row["harmonized_metric"].strip()
    )
]

add_stage3e_check(
    "S3E007",
    "Project-metric evidence retains a harmonized locked metric",
    not project_metric_role_violations,
    "blocking",
    f"Violations={project_metric_role_violations}",
    "Only defensibly mapped observations may enter project-metric analysis."
)

# S3E008 — contextual rows remain contextual.
context_role_violations = [
    row["observation_id"]
    for row in processed_rows
    if (
        row["semantic_eligibility"] == "context_only"
        and row["observation_analysis_role"] != "context_evidence"
    )
]

add_stage3e_check(
    "S3E008",
    "Context-only observations remain contextual in the processed layer",
    not context_role_violations,
    "blocking",
    f"Violations={context_role_violations}",
    "Context evidence must not become a numeric unit-economics input."
)

# S3E009 — source-defined rows remain source-defined.
source_defined_role_violations = [
    row["observation_id"]
    for row in processed_rows
    if (
        row["semantic_eligibility"] == "source_defined_only"
        and row["observation_analysis_role"]
        != "source_defined_descriptive_evidence"
    )
]

add_stage3e_check(
    "S3E009",
    "Source-defined observations remain descriptive source evidence",
    not source_defined_role_violations,
    "blocking",
    f"Violations={source_defined_role_violations}",
    "Ambiguous or source-defined metrics must not be converted into locked project metrics."
)

# S3E010 — unresolved denominators preserved.
processed_unresolved_denominators = sum(
    row["denominator_status"]
    == "metric_specific_denominator_unresolved"
    for row in processed_rows
)

add_stage3e_check(
    "S3E010",
    "Unresolved metric-specific denominators remain explicit in processed data",
    processed_unresolved_denominators
    == unresolved_denominator_count,
    "non_blocking",
    (
        f"Stage 3B unresolved={unresolved_denominator_count}; "
        f"Stage 3E unresolved={processed_unresolved_denominators}"
    ),
    "Respondent-count analyses must continue to exclude or caveat observations with unresolved denominators.",
    caveat=True,
)

# S3E011 — comparison eligibility full coverage.
add_stage3e_check(
    "S3E011",
    "Every Stage 2 comparison receives one Stage 3E analytical-eligibility decision",
    Counter(
        row["comparison_id"]
        for row in analysis_eligibility_rows
    ) == Counter(
        row["comparison_id"]
        for row in stage2_comparability
    ),
    "blocking",
    (
        f"Stage 2 comparisons={len(stage2_comparability)}; "
        f"Stage 3E decisions={len(analysis_eligibility_rows)}"
    ),
    "Cross-source analytical use must remain explicitly governed by comparison-level eligibility."
)

# S3E012 — three inflation-required comparisons not eligible for primary transformed comparison.
deferred_primary_ids = {
    row["comparison_id"]
    for row in analysis_eligibility_rows
    if row["primary_analysis_eligibility"]
    == "not_eligible_for_primary_transformed_comparison"
}

add_stage3e_check(
    "S3E012",
    "Deferred inflation comparisons remain unavailable for primary transformed analysis",
    deferred_primary_ids == {"S2C001", "S2C002", "S2C005"},
    "non_blocking",
    f"Deferred primary comparison IDs={sorted(deferred_primary_ids)}",
    "Primary analytical use must not present real-value comparisons for these uses without a separately validated transformation.",
    caveat=True,
)

# S3E013 — no direct comparison invented.
direct_eligibility_rows = [
    row["comparison_id"]
    for row in analysis_eligibility_rows
    if row["primary_analysis_eligibility"]
    == "eligible_direct_comparison"
]

add_stage3e_check(
    "S3E013",
    "Stage 3E invents no directly comparable cross-source use",
    not direct_eligibility_rows,
    "non_blocking",
    f"Direct comparison IDs={direct_eligibility_rows}",
    "All cross-source analysis must retain transformation, caveat, contextual restriction, or exclusion.",
    caveat=True,
)

# S3E014 — not-comparable uses remain excluded.
not_comparable_mismatches = [
    row["comparison_id"]
    for row in analysis_eligibility_rows
    if (
        row["stage2_comparability_status"] == "not_comparable"
        and row["primary_analysis_eligibility"] != "not_comparable"
    )
]

add_stage3e_check(
    "S3E014",
    "Stage 2 not-comparable uses remain not comparable",
    not not_comparable_mismatches,
    "blocking",
    f"Violations={not_comparable_mismatches}",
    "Stage 3E may not reopen comparisons rejected on semantic or statistical grounds."
)

# S3E015 — context-only comparison uses remain contextual.
context_comparison_mismatches = [
    row["comparison_id"]
    for row in analysis_eligibility_rows
    if (
        row["stage2_comparability_status"] == "context_only"
        and row["primary_analysis_eligibility"] != "context_only"
    )
]

add_stage3e_check(
    "S3E015",
    "Stage 2 context-only comparison uses remain contextual",
    not context_comparison_mismatches,
    "blocking",
    f"Violations={context_comparison_mismatches}",
    "Context-only evidence cannot become a numerical cross-source comparison."
)

# S3E016 — no project net metric.
project_net_processed_rows = [
    row["observation_id"]
    for row in processed_rows
    if row["harmonized_metric"] in {
        "net_operating_earnings_cash_basis",
        "net_operating_earnings_economic_basis",
    }
]

add_stage3e_check(
    "S3E016",
    "Processed data contains no manufactured project net operating earnings",
    not project_net_processed_rows,
    "blocking",
    f"Project net processed rows={project_net_processed_rows}",
    "The evidence still lacks a complete same-observation receipts-and-cost chain."
)

# S3E017 — standardized layer still unchanged.
stage3e_changed_sources = [
    source_id
    for source_id, path in source_paths.items()
    if sha256_file(path) != initial_hashes[source_id]
]

add_stage3e_check(
    "S3E017",
    "Standardized source files remain byte-identical through Stage 3E",
    not stage3e_changed_sources,
    "blocking",
    f"Changed source IDs={stage3e_changed_sources}",
    "Processed analytical preparation must not mutate the standardized evidence layer."
)

# S3E018 — Stage 3D gate remains eligible.
stage3d_summary_rows = read_csv(
    METADATA_DIR / "stage3_transformation_validation_summary.csv"
)

stage3d_ok = (
    len(stage3d_summary_rows) == 1
    and stage3d_summary_rows[0]["status"] == "PASS_WITH_CAVEAT"
    and int(stage3d_summary_rows[0]["blocking_failure_count"]) == 0
)

add_stage3e_check(
    "S3E018",
    "Stage 3D transformation gate remains eligible",
    stage3d_ok,
    "blocking",
    (
        stage3d_summary_rows[0]["status"]
        if stage3d_summary_rows
        else "missing"
    ),
    "Processed analytical outputs may only be created after a valid transformation-eligibility gate."
)

# S3E019 — output manifest matches files.
manifest_mismatches = []
for item in stage3e_output_manifest:
    path = REPO_DIR / item["output_path"]
    if not path.is_file():
        manifest_mismatches.append(
            f"{item['output_id']}:missing"
        )
        continue

    actual_count = len(read_csv(path))
    if actual_count != item["row_count"]:
        manifest_mismatches.append(
            f"{item['output_id']}:expected={item['row_count']};actual={actual_count}"
        )

add_stage3e_check(
    "S3E019",
    "Stage 3E output manifest reproduces persisted row counts",
    not manifest_mismatches,
    "blocking",
    f"Mismatches={manifest_mismatches}",
    "Stage 3F closure requires reproducible processed-output counts."
)

stage3e_status_counts = Counter(
    row["status"]
    for row in stage3e_validation
)

stage3e_blocking_failures = [
    row["check_id"]
    for row in stage3e_validation
    if (
        row["status"] == "FAIL"
        and row["severity"] == "blocking"
    )
]

stage3e_overall_status = (
    "FAIL"
    if stage3e_blocking_failures
    else "PASS_WITH_CAVEAT"
    if stage3e_status_counts.get("CAVEAT", 0) > 0
    else "PASS"
)


stage3e_validation_path = (
    METADATA_DIR / "stage3_processing_validation.csv"
)

write_csv(
    stage3e_validation_path,
    stage3e_validation,
    [
        "check_id",
        "check_name",
        "status",
        "severity",
        "evidence",
        "analytical_implication",
    ],
)

print("Stage 3E processed-output validation completed.")
print(f"Checks: {len(stage3e_validation)}")
print(f"PASS: {stage3e_status_counts.get('PASS', 0)}")
print(f"CAVEAT: {stage3e_status_counts.get('CAVEAT', 0)}")
print(f"FAIL: {stage3e_status_counts.get('FAIL', 0)}")
print(f"Overall status: {stage3e_overall_status}")


Stage 3E processed-output validation completed.
Checks: 19
PASS: 16
CAVEAT: 3
FAIL: 0
Overall status: PASS_WITH_CAVEAT


## 19. Processed-data summary

Summarize processed-data validation results and analytical evidence counts.

In [19]:

stage3e_summary_rows = [{
    "stage": "Stage 3E",
    "stage_title": "Analytical Eligibility and Processed Outputs",
    "status": stage3e_overall_status,
    "processed_observation_count": len(processed_rows),
    "comparison_eligibility_count": len(analysis_eligibility_rows),
    "output_manifest_count": len(stage3e_output_manifest),
    "validation_check_count": len(stage3e_validation),
    "validation_pass_count": stage3e_status_counts.get("PASS", 0),
    "validation_caveat_count": stage3e_status_counts.get("CAVEAT", 0),
    "validation_fail_count": stage3e_status_counts.get("FAIL", 0),
    "blocking_failure_count": len(stage3e_blocking_failures),
    "key_conclusion": (
        "The processed evidence register preserves all 72 standardized observations and source values. "
        "Observation-level semantic eligibility and comparison-level analytical eligibility remain explicit, "
        "while unsupported inflation transformations and project net operating earnings remain absent."
    ),
}]

stage3e_summary_path = (
    METADATA_DIR / "stage3_processing_validation_summary.csv"
)

write_csv(
    stage3e_summary_path,
    stage3e_summary_rows,
    [
        "stage",
        "stage_title",
        "status",
        "processed_observation_count",
        "comparison_eligibility_count",
        "output_manifest_count",
        "validation_check_count",
        "validation_pass_count",
        "validation_caveat_count",
        "validation_fail_count",
        "blocking_failure_count",
        "key_conclusion",
    ],
)

expected_stage3e_outputs = [
    processed_path,
    analysis_eligibility_path,
    stage3e_manifest_path,
    stage3e_validation_path,
    stage3e_summary_path,
]

missing_stage3e_outputs = [
    str(path.relative_to(REPO_DIR))
    for path in expected_stage3e_outputs
    if not path.is_file()
]

if missing_stage3e_outputs:
    raise RuntimeError(
        "Missing Stage 3E outputs: "
        + ", ".join(missing_stage3e_outputs)
    )

print("\n========================================")
print("STAGE 3E VALIDATION RESULTS")
print("========================================")
print(f"Status: {stage3e_overall_status}")
print(f"Checks: {len(stage3e_validation)}")
print(f"PASS: {stage3e_status_counts.get('PASS', 0)}")
print(f"CAVEAT: {stage3e_status_counts.get('CAVEAT', 0)}")
print(f"FAIL: {stage3e_status_counts.get('FAIL', 0)}")
print(f"Blocking failures: {len(stage3e_blocking_failures)}")

print("\nOutputs:")
for path in expected_stage3e_outputs:
    print(f"- {path.relative_to(REPO_DIR)}")

print("\nProcessed observation roles:")
for role, count in sorted(
    Counter(
        row["observation_analysis_role"]
        for row in processed_rows
    ).items()
):
    print(f"- {role}: {count}")

print("\nPrimary analysis eligibility:")
for status, count in sorted(
    analysis_eligibility_counts.items()
):
    print(f"- {status}: {count}")

if stage3e_blocking_failures:
    raise RuntimeError(
        "Stage 3E blocking validation failures: "
        + ", ".join(stage3e_blocking_failures)
    )

print("\nStage 3E validation summary written.")
print("Processed analytical evidence register created.")
print("No unsupported primary inflation adjustment was created.")
print("No project net operating earnings were created.")



STAGE 3E VALIDATION RESULTS
Status: PASS_WITH_CAVEAT
Checks: 19
PASS: 16
CAVEAT: 3
FAIL: 0
Blocking failures: 0

Outputs:
- data/processed/driver_evidence_processed.csv
- metadata/stage3_analysis_eligibility.csv
- metadata/stage3_output_manifest.csv
- metadata/stage3_processing_validation.csv
- metadata/stage3_processing_validation_summary.csv

Processed observation roles:
- context_evidence: 14
- project_metric_evidence: 16
- source_defined_descriptive_evidence: 42

Primary analysis eligibility:
- context_only: 3
- eligible_with_caveat: 4
- not_comparable: 7
- not_eligible_for_primary_transformed_comparison: 3

Stage 3E validation summary written.
Processed analytical evidence register created.
No unsupported primary inflation adjustment was created.
No project net operating earnings were created.


# Final Validation and Closure

Verify consistency across standardized evidence, semantic harmonization, transformation eligibility, and processed analytical outputs.

## 20. Methodological traceability

Document the methodological rationale, evidence basis, analytical implications, and report relevance of material Stage 3 rules.

In [20]:

stage3_decisions = [
    {
        "decision_id": "S3-MD001",
        "stage": "Stage 3B",
        "decision": "Preserve standardized source observations without cleaning by deletion or value replacement.",
        "rationale": "Stage 3 cleaning is an integrity and eligibility process, not a license to overwrite source evidence.",
        "evidence_basis": "Stage 2 standardized files and Stage 3B validation.",
        "analytical_implication": "All 72 standardized observations remain available with explicit eligibility and caveats.",
        "intended_report_destination": "Methodology — cleaning and evidence preservation",
    },
    {
        "decision_id": "S3-MD002",
        "stage": "Stage 3B",
        "decision": "Keep unresolved metric-specific denominators missing rather than substituting source sample size.",
        "rationale": "A source-level sample size is not automatically the denominator for every reported metric.",
        "evidence_basis": "Stage 2 validation S2V025 and Stage 3B S3B020.",
        "analytical_implication": "Analyses requiring respondent counts must exclude or caveat affected observations.",
        "intended_report_destination": "Methodology — denominators and missingness",
    },
    {
        "decision_id": "S3-MD003",
        "stage": "Stage 3B",
        "decision": "Preserve the SRC013 62-versus-67 locality discrepancy.",
        "rationale": "The source contains internally inconsistent geography counts and Stage 3 has no authoritative basis to resolve them silently.",
        "evidence_basis": "SRC013 source evidence; Stage 2 and Stage 3B validations.",
        "analytical_implication": "No single locality count is asserted as certain.",
        "intended_report_destination": "Methodology — source limitations",
    },
    {
        "decision_id": "S3-MD004",
        "stage": "Stage 3C",
        "decision": "Retain source-defined and contextual concepts when they do not match the locked project measurement framework.",
        "rationale": "Generic income, source-defined net, mixed cost bundles, perception/preference, generic working time, and ambiguous distance are not interchangeable with project unit-economics metrics.",
        "evidence_basis": "Stage 0 measurement dictionary; Stage 2 semantic safeguards; Stage 3C harmonization assessment.",
        "analytical_implication": "Only defensibly mapped observations enter project-metric analysis; other observations remain descriptive or contextual.",
        "intended_report_destination": "Methodology — semantic harmonization",
    },
    {
        "decision_id": "S3-MD005",
        "stage": "Stage 3C",
        "decision": "Keep driver-reported deduction evidence separate from realized transaction deductions and regulatory/platform fee layers.",
        "rationale": "These evidence layers answer different questions and are not numerically interchangeable.",
        "evidence_basis": "Stage 0 deduction-layer rules and Stage 2 comparability assessment.",
        "analytical_implication": "Reported deduction prevalence cannot be used as a realized commission rate.",
        "intended_report_destination": "Methodology — deduction evidence layers",
    },
    {
        "decision_id": "S3-MD006",
        "stage": "Stage 3D",
        "decision": "Preserve nominal monetary values and defer primary inflation adjustment for S2C001, S2C002, and S2C005.",
        "rationale": "The currently registered CPI evidence does not provide a sufficiently matched price-period, geography, and base-regime bridge for a primary transformation.",
        "evidence_basis": "Stage 2 comparability assessment; Stage 3 transformation reference registry and decision log.",
        "analytical_implication": "No primary real-value comparison is produced for the three inflation-required uses.",
        "intended_report_destination": "Methodology — monetary transformation eligibility",
    },
    {
        "decision_id": "S3-MD007",
        "stage": "Stage 3D",
        "decision": "Do not ratio CPI index levels directly across incompatible base-year regimes.",
        "rationale": "BPS CPI base and geographic coverage changed across the relevant periods.",
        "evidence_basis": "Stage 3 transformation reference registry.",
        "analytical_implication": "Any later sensitivity transformation requires explicit CPI source, geography, period, formula, and caveat metadata.",
        "intended_report_destination": "Methodology — CPI compatibility",
    },
    {
        "decision_id": "S3-MD008",
        "stage": "Stage 3E",
        "decision": "Create a source-preserving processed evidence register rather than a pooled estimate table.",
        "rationale": "The evidence contains heterogeneous definitions, sample frames, geographies, and comparability classes.",
        "evidence_basis": "Stage 3C semantic eligibility and Stage 3E analytical eligibility.",
        "analytical_implication": "All observations remain traceable while inappropriate cross-source pooling is blocked.",
        "intended_report_destination": "Methodology — analytical dataset construction",
    },
    {
        "decision_id": "S3-MD009",
        "stage": "Stage 3E",
        "decision": "Keep comparability specific to a defined analytical use.",
        "rationale": "The same observation may be usable for one analytical question and unusable for another.",
        "evidence_basis": "Stage 2 comparability assessment and Stage 3E analysis eligibility.",
        "analytical_implication": "No global comparability label is attached to processed observations.",
        "intended_report_destination": "Methodology — comparability framework",
    },
    {
        "decision_id": "S3-MD010",
        "stage": "Stage 3E",
        "decision": "Do not construct observed project net operating earnings from incomplete source components.",
        "rationale": "The evidence lacks a complete same-observation chain of gross earnings, realized deductions, receipts, and project-compatible operating costs.",
        "evidence_basis": "Stage 2 S2V031, Stage 3B S3B023, and Stage 3E validation.",
        "analytical_implication": "Later analytical use may assess available components or scenarios, but must not represent them as an observed full gross-to-net chain.",
        "intended_report_destination": "Methodology — unit-economics reconstruction limits",
    },
]

stage3_decision_log_path = (
    METADATA_DIR / "stage3_methodological_decision_log.csv"
)

write_csv(
    stage3_decision_log_path,
    stage3_decisions,
    [
        "decision_id",
        "stage",
        "decision",
        "rationale",
        "evidence_basis",
        "analytical_implication",
        "intended_report_destination",
    ],
)

print(f"Wrote {stage3_decision_log_path.relative_to(REPO_DIR)}")
print(f"Material Stage 3 decisions: {len(stage3_decisions)}")


Wrote metadata/stage3_methodological_decision_log.csv
Material Stage 3 decisions: 10


## 21. Cross-layer validation

Validate observation lineage, comparison coverage, denominator limitations, geography caveats, transformation restrictions, and unit-economics measurement boundaries.

In [21]:

stage3_final_validation = []

def add_stage3f_check(
    check_id,
    check_name,
    condition,
    severity,
    evidence,
    implication,
    caveat=False,
):
    status = (
        "CAVEAT"
        if condition and caveat
        else "PASS"
        if condition
        else "FAIL"
    )

    stage3_final_validation.append({
        "check_id": check_id,
        "check_name": check_name,
        "status": status,
        "severity": severity,
        "evidence": evidence,
        "analytical_implication": implication,
    })

# Load persisted Stage 3 summaries.
stage3b_summary = read_csv(
    METADATA_DIR / "stage3_cleaning_validation_summary.csv"
)
stage3c_summary = read_csv(
    METADATA_DIR / "stage3_harmonization_validation_summary.csv"
)
stage3d_summary = read_csv(
    METADATA_DIR / "stage3_transformation_validation_summary.csv"
)
stage3e_summary = read_csv(
    METADATA_DIR / "stage3_processing_validation_summary.csv"
)

# S3F001 — Stage 2 baseline.
add_stage3f_check(
    "S3F001",
    "Stage 2 baseline remains the approved source of truth",
    (
        branch == "main"
        and local_head == EXPECTED_STAGE2_HEAD
        and remote_head == EXPECTED_STAGE2_HEAD
    ),
    "blocking",
    (
        f"branch={branch}; local={local_head}; remote={remote_head}; "
        f"expected={EXPECTED_STAGE2_HEAD}"
    ),
    "Stage 3 closure is valid only against the approved Stage 2 baseline."
)

# S3F002 — all substage summaries exist and remain eligible.
summary_gate_rows = [
    ("Stage 3B", stage3b_summary),
    ("Stage 3C", stage3c_summary),
    ("Stage 3D", stage3d_summary),
    ("Stage 3E", stage3e_summary),
]

summary_gate_failures = []
for stage_name, rows in summary_gate_rows:
    if (
        len(rows) != 1
        or rows[0]["status"] != "PASS_WITH_CAVEAT"
        or int(rows[0]["blocking_failure_count"]) != 0
    ):
        summary_gate_failures.append(stage_name)

add_stage3f_check(
    "S3F002",
    "Stages 3B through 3E remain valid PASS_WITH_CAVEAT gates",
    not summary_gate_failures,
    "blocking",
    f"Invalid substage summaries={summary_gate_failures}",
    "Final Stage 3 closure requires every prior Stage 3 gate to remain valid."
)

# S3F003 — processed count.
persisted_processed_rows = read_csv(processed_path)

add_stage3f_check(
    "S3F003",
    "Processed evidence register preserves all 72 standardized observations",
    len(persisted_processed_rows) == expected_observation_count,
    "blocking",
    (
        f"Processed observations={len(persisted_processed_rows)}; "
        f"expected={expected_observation_count}"
    ),
    "The processed analytical layer must preserve the complete source-preserving evidence register."
)

# S3F004 — observation ID identity.
persisted_processed_ids = {
    row["observation_id"]
    for row in persisted_processed_rows
}

add_stage3f_check(
    "S3F004",
    "Processed observation identities exactly match the standardized evidence",
    persisted_processed_ids == standardized_ids,
    "blocking",
    (
        f"Missing={sorted(standardized_ids - persisted_processed_ids)}; "
        f"unexpected={sorted(persisted_processed_ids - standardized_ids)}"
    ),
    "No evidence may be dropped or synthetically introduced between standardized and processed layers."
)

# S3F005 — source values preserved.
persisted_processed_by_id = {
    row["observation_id"]: row
    for row in persisted_processed_rows
}

final_value_violations = []
for observation_id, source in standardized_by_id.items():
    processed = persisted_processed_by_id[observation_id]

    if (
        processed["value_numeric"] != source["value_numeric"]
        or processed["source_value_text"] != source["source_value_text"]
        or processed["unit"] != source["unit"]
        or processed["processed_value_numeric"] != source["value_numeric"]
        or processed["processed_unit"] != source["unit"]
    ):
        final_value_violations.append(observation_id)

add_stage3f_check(
    "S3F005",
    "Processed evidence remains source-value preserving",
    not final_value_violations,
    "blocking",
    f"Value-preservation violations={final_value_violations}",
    "Later analytical use must distinguish source evidence from derived or modelled values."
)

# S3F006 — no missing-to-zero conversion.
missing_to_zero_violations = []
for observation_id, source in standardized_by_id.items():
    processed = persisted_processed_by_id[observation_id]

    source_numeric_missing = not source["value_numeric"].strip()
    processed_numeric = processed["processed_value_numeric"].strip()

    if source_numeric_missing and processed_numeric == "0":
        missing_to_zero_violations.append(observation_id)

add_stage3f_check(
    "S3F006",
    "Missing standardized values are never converted to zero",
    not missing_to_zero_violations,
    "blocking",
    f"Violations={missing_to_zero_violations}",
    "Missing evidence must remain missing."
)

# S3F007 — Stage 3C semantic distribution.
final_semantic_counts = Counter(
    row["semantic_eligibility"]
    for row in persisted_processed_rows
)

expected_semantic_counts = Counter({
    "context_only": 14,
    "eligible_as_mapped": 6,
    "eligible_with_semantic_caveat": 10,
    "source_defined_only": 42,
})

add_stage3f_check(
    "S3F007",
    "Final processed semantic eligibility matches the validated Stage 3C distribution",
    final_semantic_counts == expected_semantic_counts,
    "blocking",
    (
        f"Observed={dict(final_semantic_counts)}; "
        f"expected={dict(expected_semantic_counts)}"
    ),
    "Semantic eligibility must not change after Stage 3C."
)

# S3F008 — Stage 3E role distribution.
final_role_counts = Counter(
    row["observation_analysis_role"]
    for row in persisted_processed_rows
)

expected_role_counts = Counter({
    "context_evidence": 14,
    "project_metric_evidence": 16,
    "source_defined_descriptive_evidence": 42,
})

add_stage3f_check(
    "S3F008",
    "Final processed analysis roles match the validated Stage 3E distribution",
    final_role_counts == expected_role_counts,
    "blocking",
    (
        f"Observed={dict(final_role_counts)}; "
        f"expected={dict(expected_role_counts)}"
    ),
    "Later analytical use must retain the locked observation-level analytical roles."
)

# S3F009 — comparison-level eligibility distribution.
persisted_analysis_eligibility = read_csv(
    analysis_eligibility_path
)

final_analysis_eligibility_counts = Counter(
    row["primary_analysis_eligibility"]
    for row in persisted_analysis_eligibility
)

expected_analysis_eligibility_counts = Counter({
    "context_only": 3,
    "eligible_with_caveat": 4,
    "not_comparable": 7,
    "not_eligible_for_primary_transformed_comparison": 3,
})

add_stage3f_check(
    "S3F009",
    "Final comparison-level analytical eligibility matches Stage 3E",
    final_analysis_eligibility_counts
    == expected_analysis_eligibility_counts,
    "blocking",
    (
        f"Observed={dict(final_analysis_eligibility_counts)}; "
        f"expected={dict(expected_analysis_eligibility_counts)}"
    ),
    "Later analytical use must honor the locked use-specific comparability and transformation restrictions."
)

# S3F010 — all 17 comparison IDs preserved.
final_comparison_ids = {
    row["comparison_id"]
    for row in persisted_analysis_eligibility
}

expected_comparison_ids = {
    row["comparison_id"]
    for row in stage2_comparability
}

add_stage3f_check(
    "S3F010",
    "All 17 Stage 2 comparison uses remain represented",
    (
        len(persisted_analysis_eligibility)
        == expected_comparison_count
        and final_comparison_ids == expected_comparison_ids
    ),
    "blocking",
    (
        f"Rows={len(persisted_analysis_eligibility)}; "
        f"expected={expected_comparison_count}"
    ),
    "No comparison-use decision may disappear during Stage 3."
)

# S3F011 — deferred inflation uses.
final_deferred_ids = {
    row["comparison_id"]
    for row in persisted_analysis_eligibility
    if row["primary_analysis_eligibility"]
    == "not_eligible_for_primary_transformed_comparison"
}

add_stage3f_check(
    "S3F011",
    "The three inflation-required comparisons remain deferred for primary transformed analysis",
    final_deferred_ids == {"S2C001", "S2C002", "S2C005"},
    "non_blocking",
    f"Deferred comparison IDs={sorted(final_deferred_ids)}",
    "Primary analytical use must not present inflation-adjusted level comparisons for these uses.",
    caveat=True,
)

# S3F012 — unresolved denominators.
final_unresolved_denominators = sum(
    row["denominator_status"]
    == "metric_specific_denominator_unresolved"
    for row in persisted_processed_rows
)

add_stage3f_check(
    "S3F012",
    "Unresolved metric-specific denominators remain explicit",
    final_unresolved_denominators
    == unresolved_denominator_count,
    "non_blocking",
    (
        f"Stage 3B unresolved={unresolved_denominator_count}; "
        f"final processed unresolved={final_unresolved_denominators}"
    ),
    "Respondent-count analyses must continue to exclude or caveat these rows.",
    caveat=True,
)

# S3F013 — SRC013 geography discrepancy retained.
src013_processed_text = " ".join(
    " ".join([
        row.get("geography", ""),
        row.get("extraction_notes", ""),
    ])
    for row in persisted_processed_rows
    if row["source_id"] == "SRC013"
)

add_stage3f_check(
    "S3F013",
    "SRC013 62-versus-67 locality discrepancy remains explicit",
    "62" in src013_processed_text and "67" in src013_processed_text,
    "non_blocking",
    "Both locality counts remain present in processed SRC013 evidence.",
    "No single locality count may be asserted as certain.",
    caveat=True,
)

# S3F014 — no direct comparison.
final_direct_comparisons = [
    row["comparison_id"]
    for row in persisted_analysis_eligibility
    if row["primary_analysis_eligibility"]
    == "eligible_direct_comparison"
]

add_stage3f_check(
    "S3F014",
    "No cross-source comparison becomes directly comparable during Stage 3",
    not final_direct_comparisons,
    "non_blocking",
    f"Direct comparison IDs={final_direct_comparisons}",
    "All cross-source uses retain transformation, caveat, contextual restriction, or exclusion.",
    caveat=True,
)

# S3F015 — no project net.
final_project_net_rows = [
    row["observation_id"]
    for row in persisted_processed_rows
    if row["harmonized_metric"] in {
        "net_operating_earnings_cash_basis",
        "net_operating_earnings_economic_basis",
    }
]

add_stage3f_check(
    "S3F015",
    "Stage 3 contains no manufactured project net operating earnings",
    not final_project_net_rows,
    "non_blocking",
    f"Project net rows={final_project_net_rows}",
    "Later analytical use may assess components or scenarios but must not claim an observed complete gross-to-net chain.",
    caveat=True,
)

# S3F016 — no unsupported transformed rows.
final_transformed_rows = [
    row["observation_id"]
    for row in persisted_processed_rows
    if (
        row["processed_transformation_status"] != "none"
        or row["processed_transformation_formula"].strip()
    )
]

add_stage3f_check(
    "S3F016",
    "No unsupported transformed primary values exist in the processed layer",
    not final_transformed_rows,
    "blocking",
    f"Transformed processed rows={final_transformed_rows}",
    "Deferred transformations must remain absent from the primary analytical evidence."
)

# S3F017 — standardized byte immutability.
final_changed_sources = [
    source_id
    for source_id, path in source_paths.items()
    if sha256_file(path) != initial_hashes[source_id]
]

add_stage3f_check(
    "S3F017",
    "All five Stage 2 standardized source files remain byte-identical",
    not final_changed_sources,
    "blocking",
    f"Changed source IDs={final_changed_sources}",
    "Stage 3 must preserve the standardized evidence layer exactly."
)

# S3F018 — output manifest.
persisted_stage3e_manifest = read_csv(
    stage3e_manifest_path
)

manifest_errors = []
for item in persisted_stage3e_manifest:
    path = REPO_DIR / item["output_path"]
    if not path.is_file():
        manifest_errors.append(
            f"{item['output_id']}:missing"
        )
        continue

    actual_count = len(read_csv(path))
    if actual_count != int(item["row_count"]):
        manifest_errors.append(
            f"{item['output_id']}:expected={item['row_count']};actual={actual_count}"
        )

add_stage3f_check(
    "S3F018",
    "Stage 3E output manifest reproduces persisted analytical outputs",
    not manifest_errors,
    "blocking",
    f"Manifest errors={manifest_errors}",
    "Stage 3 closure requires reproducible processed-output counts."
)

# S3F019 — methodological decision traceability.
add_stage3f_check(
    "S3F019",
    "Material Stage 3 methodological decisions are explicitly logged",
    len(stage3_decisions) >= 10,
    "blocking",
    f"Logged decisions={len(stage3_decisions)}",
    "Material decisions must remain traceable into later methodology and reporting."
)

# S3F020 — source→processed lineage.
lineage_errors = []
for row in persisted_processed_rows:
    observation_id = row["observation_id"]
    source_id = row["source_id"]

    if (
        observation_id not in standardized_by_id
        or source_id != standardized_by_id[observation_id]["source_id"]
    ):
        lineage_errors.append(observation_id)

add_stage3f_check(
    "S3F020",
    "Source-to-standardized-to-processed lineage is complete",
    not lineage_errors,
    "blocking",
    f"Lineage errors={lineage_errors}",
    "Every downstream analytical observation must resolve back to its standardized source observation."
)

# S3F021 — no blocking failures across Stage 3 validation files.
stage3_validation_files = [
    METADATA_DIR / "stage3_cleaning_validation.csv",
    METADATA_DIR / "stage3_harmonization_validation.csv",
    METADATA_DIR / "stage3_transformation_validation.csv",
    METADATA_DIR / "stage3_processing_validation.csv",
]

prior_stage3_blocking_failures = []

for path in stage3_validation_files:
    for row in read_csv(path):
        if (
            row["status"] == "FAIL"
            and row["severity"] == "blocking"
        ):
            prior_stage3_blocking_failures.append(
                f"{path.name}:{row['check_id']}"
            )

add_stage3f_check(
    "S3F021",
    "No blocking Stage 3B–3E validation failure remains",
    not prior_stage3_blocking_failures,
    "blocking",
    f"Blocking failures={prior_stage3_blocking_failures}",
    "Stage 3 closure requires all blocking validation failures to be resolved."
)

stage3f_status_counts = Counter(
    row["status"]
    for row in stage3_final_validation
)

stage3f_blocking_failures = [
    row["check_id"]
    for row in stage3_final_validation
    if (
        row["status"] == "FAIL"
        and row["severity"] == "blocking"
    )
]

stage3_closure_status = (
    "FAIL"
    if stage3f_blocking_failures
    else "PASS_WITH_CAVEAT"
    if stage3f_status_counts.get("CAVEAT", 0) > 0
    else "PASS"
)


stage3_final_validation_path = (
    METADATA_DIR / "stage3_final_validation.csv"
)

write_csv(
    stage3_final_validation_path,
    stage3_final_validation,
    [
        "check_id",
        "check_name",
        "status",
        "severity",
        "evidence",
        "analytical_implication",
    ],
)

print("Stage 3 final validation completed.")
print(f"Checks: {len(stage3_final_validation)}")
print(f"PASS: {stage3f_status_counts.get('PASS', 0)}")
print(f"CAVEAT: {stage3f_status_counts.get('CAVEAT', 0)}")
print(f"FAIL: {stage3f_status_counts.get('FAIL', 0)}")
print(f"Closure status: {stage3_closure_status}")


Stage 3 final validation completed.
Checks: 21
PASS: 16
CAVEAT: 5
FAIL: 0
Closure status: PASS_WITH_CAVEAT


## 22. Stage 3 closure

Summarize the final validation results, material evidence limitations, and closure status of the Stage 3 analytical evidence base.


In [23]:

stage3_closure_rows = [{
    "stage": "Stage 3",
    "stage_title": "Data Cleaning, Harmonization and Validation",
    "closure_status": stage3_closure_status,
    "standardized_source_file_count": len(source_paths),
    "standardized_observation_count": len(stage3_rows),
    "processed_observation_count": len(persisted_processed_rows),
    "semantic_context_only_count": final_semantic_counts.get("context_only", 0),
    "semantic_eligible_as_mapped_count": final_semantic_counts.get("eligible_as_mapped", 0),
    "semantic_eligible_with_caveat_count": final_semantic_counts.get("eligible_with_semantic_caveat", 0),
    "semantic_source_defined_only_count": final_semantic_counts.get("source_defined_only", 0),
    "comparison_eligibility_count": len(persisted_analysis_eligibility),
    "comparison_context_only_count": final_analysis_eligibility_counts.get("context_only", 0),
    "comparison_eligible_with_caveat_count": final_analysis_eligibility_counts.get("eligible_with_caveat", 0),
    "comparison_not_comparable_count": final_analysis_eligibility_counts.get("not_comparable", 0),
    "comparison_deferred_transformation_count": final_analysis_eligibility_counts.get(
        "not_eligible_for_primary_transformed_comparison", 0
    ),
    "final_validation_check_count": len(stage3_final_validation),
    "final_validation_pass_count": stage3f_status_counts.get("PASS", 0),
    "final_validation_caveat_count": stage3f_status_counts.get("CAVEAT", 0),
    "final_validation_fail_count": stage3f_status_counts.get("FAIL", 0),
    "blocking_failure_count": len(stage3f_blocking_failures),
    "key_conclusion": (
        "Stage 3 cleaning, semantic harmonization, transformation eligibility, and analytical preparation are complete. "
        "The processed evidence register preserves all 72 standardized observations and source values, "
        "with semantic, denominator, geography, transformation, comparability, and incomplete unit-economics-chain caveats retained explicitly."
    ),
    "binding_caveats": (
        "Metric-specific denominators remain unresolved for some observations. | "
        "SRC013 retains the internal 62-versus-67 locality discrepancy. | "
        "No assessed cross-source comparison is directly comparable. | "
        "Primary inflation adjustment remains deferred for S2C001, S2C002, and S2C005. | "
        "The observed evidence does not contain a complete same-observation gross-to-net unit-economics chain."
    ),
}]

stage3_closure_summary_path = (
    METADATA_DIR / "stage3_closure_summary.csv"
)

write_csv(
    stage3_closure_summary_path,
    stage3_closure_rows,
    [
        "stage",
        "stage_title",
        "closure_status",
        "standardized_source_file_count",
        "standardized_observation_count",
        "processed_observation_count",
        "semantic_context_only_count",
        "semantic_eligible_as_mapped_count",
        "semantic_eligible_with_caveat_count",
        "semantic_source_defined_only_count",
        "comparison_eligibility_count",
        "comparison_context_only_count",
        "comparison_eligible_with_caveat_count",
        "comparison_not_comparable_count",
        "comparison_deferred_transformation_count",
        "final_validation_check_count",
        "final_validation_pass_count",
        "final_validation_caveat_count",
        "final_validation_fail_count",
        "blocking_failure_count",
        "key_conclusion",
        "binding_caveats",
    ],
)

closure_validation = []

def add_closure_check(
    check_id,
    check_name,
    condition,
    evidence,
    implication,
):
    closure_validation.append({
        "check_id": check_id,
        "check_name": check_name,
        "status": "PASS" if condition else "FAIL",
        "severity": "blocking",
        "evidence": evidence,
        "analytical_implication": implication,
    })

add_closure_check(
    "S3CL001",
    "Stage 3 final validation has no blocking failure",
    not stage3f_blocking_failures,
    f"Blocking failures={stage3f_blocking_failures}",
    "Stage 3 closure requires all blocking validation failures to be resolved."
)

add_closure_check(
    "S3CL002",
    "Processed observation count equals standardized observation count",
    len(persisted_processed_rows) == len(stage3_rows) == 72,
    f"Standardized={len(stage3_rows)}; processed={len(persisted_processed_rows)}",
    "The analytical evidence register must preserve the complete Stage 2 evidence base."
)

add_closure_check(
    "S3CL003",
    "Processed source values remain unchanged",
    not final_value_violations,
    f"Value violations={final_value_violations}",
    "Stage 3 closure requires source-value preservation."
)

add_closure_check(
    "S3CL004",
    "All 17 comparison-use decisions remain traceable",
    len(persisted_analysis_eligibility) == 17,
    f"Comparison eligibility rows={len(persisted_analysis_eligibility)}",
    "Later analytical use must inherit every use-specific comparability decision."
)

add_closure_check(
    "S3CL005",
    "No unsupported transformed primary value is present",
    not final_transformed_rows,
    f"Transformed rows={final_transformed_rows}",
    "Deferred inflation transformations must remain absent from primary analysis."
)

add_closure_check(
    "S3CL006",
    "No project net operating earnings are manufactured",
    not final_project_net_rows,
    f"Project net rows={final_project_net_rows}",
    "Later analytical use must not claim an observed complete gross-to-net chain."
)

stage3_closure_validation_path = (
    METADATA_DIR / "stage3_closure_validation.csv"
)

write_csv(
    stage3_closure_validation_path,
    closure_validation,
    [
        "check_id",
        "check_name",
        "status",
        "severity",
        "evidence",
        "analytical_implication",
    ],
)

closure_fail_ids = [
    row["check_id"]
    for row in closure_validation
    if row["status"] == "FAIL"
]

print("\n========================================")
print("STAGE 3F VALIDATION RESULTS")
print("========================================")
print(f"Stage 3 closure status: {stage3_closure_status}")
print(f"Final validation checks: {len(stage3_final_validation)}")
print(f"PASS: {stage3f_status_counts.get('PASS', 0)}")
print(f"CAVEAT: {stage3f_status_counts.get('CAVEAT', 0)}")
print(f"FAIL: {stage3f_status_counts.get('FAIL', 0)}")
print(f"Blocking failures: {len(stage3f_blocking_failures)}")
print(f"Closure validation PASS: {sum(row['status'] == 'PASS' for row in closure_validation)}/{len(closure_validation)}")

stage3_research_output_paths = [
    Path("data/processed/driver_evidence_processed.csv"),
    Path("metadata/stage3_input_integrity_manifest.csv"),
    Path("metadata/stage3_cleaning_profile.csv"),
    Path("metadata/stage3_cleaning_validation.csv"),
    Path("metadata/stage3_cleaning_validation_summary.csv"),
    Path("metadata/stage3_harmonization_rules.csv"),
    Path("metadata/stage3_harmonization_assessment.csv"),
    Path("metadata/stage3_harmonization_validation.csv"),
    Path("metadata/stage3_harmonization_validation_summary.csv"),
    Path("metadata/stage3_transformation_reference_registry.csv"),
    Path("metadata/stage3_transformation_eligibility.csv"),
    Path("metadata/stage3_transformation_decision_log.csv"),
    Path("metadata/stage3_transformation_validation.csv"),
    Path("metadata/stage3_transformation_validation_summary.csv"),
    Path("metadata/stage3_analysis_eligibility.csv"),
    Path("metadata/stage3_output_manifest.csv"),
    Path("metadata/stage3_processing_validation.csv"),
    Path("metadata/stage3_processing_validation_summary.csv"),
    Path("metadata/stage3_methodological_decision_log.csv"),
    Path("metadata/stage3_final_validation.csv"),
    Path("metadata/stage3_closure_summary.csv"),
    Path("metadata/stage3_closure_validation.csv"),
]

missing_stage3_research_outputs = [
    str(path)
    for path in stage3_research_output_paths
    if not (REPO_DIR / path).is_file()
]
if missing_stage3_research_outputs:
    raise RuntimeError(
        "Missing Stage 3 research outputs: "
        + ", ".join(missing_stage3_research_outputs)
    )

print("\nStage 3 research outputs:")
for path in stage3_research_output_paths:
    print(f"- {path}")



STAGE 3F VALIDATION RESULTS
Stage 3 closure status: PASS_WITH_CAVEAT
Final validation checks: 21
PASS: 16
CAVEAT: 5
FAIL: 0
Blocking failures: 0
Closure validation PASS: 6/6

Stage 3 research outputs:
- data/processed/driver_evidence_processed.csv
- metadata/stage3_input_integrity_manifest.csv
- metadata/stage3_cleaning_profile.csv
- metadata/stage3_cleaning_validation.csv
- metadata/stage3_cleaning_validation_summary.csv
- metadata/stage3_harmonization_rules.csv
- metadata/stage3_harmonization_assessment.csv
- metadata/stage3_harmonization_validation.csv
- metadata/stage3_harmonization_validation_summary.csv
- metadata/stage3_transformation_reference_registry.csv
- metadata/stage3_transformation_eligibility.csv
- metadata/stage3_transformation_decision_log.csv
- metadata/stage3_transformation_validation.csv
- metadata/stage3_transformation_validation_summary.csv
- metadata/stage3_analysis_eligibility.csv
- metadata/stage3_output_manifest.csv
- metadata/stage3_processing_validation.cs